# RevConnectAI v5: API-Enabled GW Student Engagement Assistant

RevConnectAI combines GW student-organization policy documents, SGA guidance, structured planning tools, and a read-only CampusGroups connection. It can answer policy questions, explain CampusGroups procedures, recommend organizations and public events, route students to campus resources, support engagement planning, estimate event budgets, and coach funding applications.

**Important:** RevConnectAI provides guidance, not final funding, purchasing, contract, reimbursement, travel, event, or policy decisions. Authorized GW staff and current official sources control.

## Notebook version: **v5.0 API Prototype**

Adds a working read-only CampusGroups RSS client for current public organizations and upcoming public events, secure credential loading, API snapshots, connection diagnostics, live search, privacy filters, and automatic fallback to the bundled knowledge base.

In [ ]:
!pip -q install sentence-transformers faiss-cpu transformers accelerate sentencepiece gradio pypdf pandas openpyxl beautifulsoup4 requests urllib3

In [ ]:
import os
import re
import json
import html
import hashlib
import zipfile
import warnings
import requests
import xml.etree.ElementTree as ET
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import faiss
import numpy as np
import pandas as pd
import torch

from bs4 import BeautifulSoup
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

warnings.filterwarnings("ignore")

PROJECT_DIR = Path("/content/revconnect_ai")
KNOWLEDGE_DIR = PROJECT_DIR / "knowledge_base"
CACHE_DIR = PROJECT_DIR / "cache"
KNOWLEDGE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
LOCAL_GENERATION_MODEL = "google/flan-t5-base"

# Enable the read-only CampusGroups connection. The notebook falls back safely when no credential is available.
USE_CAMPUSGROUPS_API = True

print("Knowledge folder:", KNOWLEDGE_DIR)

## 1. Load the bundled knowledge base

Run the next cell. In Colab, upload `RevConnectAI_Knowledge_Base_v4.zip` when prompted. The full project bundle is also recognized automatically.

In [ ]:
def extract_zip_recursive(zip_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(destination)
    for nested_zip in list(destination.rglob("*.zip")):
        marker = nested_zip.with_suffix(nested_zip.suffix + ".extracted")
        if marker.exists():
            continue
        try:
            nested_destination = nested_zip.parent / nested_zip.stem
            nested_destination.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(nested_zip, "r") as archive:
                archive.extractall(nested_destination)
            marker.write_text("extracted", encoding="utf-8")
        except zipfile.BadZipFile:
            pass


def copy_or_extract_source(path: Path) -> None:
    if not path.exists() or not path.is_file():
        return
    if path.suffix.lower() == ".zip":
        extract_zip_recursive(path, KNOWLEDGE_DIR)
    elif path.suffix.lower() in {".pdf", ".csv", ".xlsx", ".xls", ".txt", ".md"}:
        import shutil
        target = KNOWLEDGE_DIR / path.name
        if path.resolve() != target.resolve():
            shutil.copy2(path, target)


def auto_locate_knowledge_sources() -> List[str]:
    search_roots = [Path.cwd(), Path("/content"), Path("/mnt/data"), Path("/tmp")]
    preferred_names = [
        "RevConnectAI_Knowledge_Base_v4.zip",
        "RevConnectAI_v4_Full_Project_Bundle.zip",
        "RevConnectAI_Knowledge_Base_v3.zip",
        "Student Organization Handbook.pdf",
        "SGA Bylaws (1).pdf",
        "Finance Committee _ GW SGA.pdf",
        "Applying for Funding _ GW SGA.pdf",
        "Frequently Asked Questions _ GW SGA.pdf",
    ]
    seen = set()
    for root in search_roots:
        if not root.exists():
            continue
        for name in preferred_names:
            candidates = [root / name]
            try:
                candidates.extend(root.rglob(name))
            except Exception:
                pass
            for candidate in candidates:
                if candidate.exists() and candidate.is_file():
                    resolved = candidate.resolve()
                    if resolved not in seen:
                        seen.add(resolved)
                        copy_or_extract_source(candidate)
    return [str(path) for path in KNOWLEDGE_DIR.rglob("*") if path.is_file()]


def upload_knowledge_sources() -> List[str]:
    discovered = auto_locate_knowledge_sources()
    if list(KNOWLEDGE_DIR.rglob("*.pdf")):
        return discovered
    try:
        from google.colab import files
    except ImportError:
        print("Copy RevConnectAI_Knowledge_Base_v4.zip into:", KNOWLEDGE_DIR)
        return discovered
    uploaded = files.upload()
    for filename, content in uploaded.items():
        temporary = Path("/content") / Path(filename).name
        temporary.write_bytes(content)
        copy_or_extract_source(temporary)
    return auto_locate_knowledge_sources()

uploaded_paths = upload_knowledge_sources()
print("PDFs available:")
for pdf in sorted(KNOWLEDGE_DIR.rglob("*.pdf")):
    print("-", pdf.name)

## 2. Source registry

The registry preserves source scope, authority, update date, and verification link.

In [ ]:
SOURCE_REGISTRY = {
    "Student Organization Handbook.pdf": {
        "title": "GW Student Organization Handbook",
        "source_type": "University Policy",
        "authority_rank": 100,
        "source_url": "https://students.gwu.edu/org-handbook",
        "document_updated": "2025-03",
    },
    "SGA Bylaws (1).pdf": {
        "title": "GW Student Government Association Bylaws",
        "source_type": "SGA Governing Document",
        "authority_rank": 95,
        "source_url": "",
        "document_updated": "",
    },
    "Finance Committee _ GW SGA.pdf": {
        "title": "GW SGA Finance Committee",
        "source_type": "SGA Guidance",
        "authority_rank": 85,
        "source_url": "https://sga.gwu.edu/finance-committee",
        "document_updated": "2026-07-20",
    },
    "Applying for Funding _ GW SGA.pdf": {
        "title": "GW SGA Applying for Funding",
        "source_type": "SGA Guidance",
        "authority_rank": 85,
        "source_url": "https://sga.gwu.edu/applying-for-funding",
        "document_updated": "2026-07-20",
    },
    "Frequently Asked Questions _ GW SGA.pdf": {
        "title": "GW SGA Frequently Asked Questions",
        "source_type": "SGA Guidance",
        "authority_rank": 85,
        "source_url": "https://sga.gwu.edu/frequently",
        "document_updated": "2026-07-20",
    },
}

PRECEDENCE_GUIDANCE = (
    "University Policy governs university requirements, conduct, events, contracts, travel, "
    "purchasing, and registration. SGA Governing Documents govern SGA procedures, allocations, "
    "committees, and SGA financial limitations. SGA Guidance explains processes but should be "
    "checked against current bylaws and university policy. CampusGroups Operational Data provides "
    "current organizations and events and does not create policy. Engagement Strategy is advice only."
)

## 3. Load every PDF page

Page-level records let the assistant cite exact pages instead of citing only the full document.

In [ ]:
def clean_text(value: Any) -> str:
    if value is None:
        return ""
    text = html.unescape(str(value))
    text = BeautifulSoup(text, "html.parser").get_text(" ", strip=True)
    text = text.replace("\u00ad", "")
    return re.sub(r"\s+", " ", text).strip()


def stable_id(*parts: Any) -> str:
    joined = "||".join(str(part) for part in parts)
    return hashlib.sha1(joined.encode("utf-8")).hexdigest()[:16]


def discover_files(directory: Path) -> List[Path]:
    supported = {".pdf", ".csv", ".xlsx", ".xls", ".txt", ".md"}
    return sorted(
        p for p in directory.rglob("*")
        if p.is_file() and p.suffix.lower() in supported
    )


def load_pdf_pages(path: Path) -> List[Dict[str, Any]]:
    registry = SOURCE_REGISTRY.get(path.name, {})
    reader = PdfReader(str(path))
    records = []

    for page_index, page in enumerate(reader.pages):
        text = clean_text(page.extract_text() or "")
        if not text:
            continue

        page_number = page_index + 1
        records.append({
            "record_id": stable_id(path.name, page_number),
            "document_title": registry.get("title", path.stem),
            "filename": path.name,
            "source_type": registry.get("source_type", "Reference Document"),
            "authority_rank": registry.get("authority_rank", 60),
            "source_url": registry.get("source_url", ""),
            "document_updated": registry.get("document_updated", ""),
            "page_number": page_number,
            "text": text,
            "contains_legacy_engage_reference": bool(
                re.search(r"\bEngage\b", text, flags=re.IGNORECASE)
            ),
        })

    return records


source_files = discover_files(KNOWLEDGE_DIR)
pdf_files = [p for p in source_files if p.suffix.lower() == ".pdf"]

page_records = []
for pdf_file in pdf_files:
    page_records.extend(load_pdf_pages(pdf_file))

pages_df = pd.DataFrame(page_records)

if pages_df.empty:
    raise RuntimeError("No PDF pages loaded. Upload the knowledge-base ZIP first.")

print(f"Loaded {len(pages_df):,} pages from {len(pdf_files)} PDFs.")
display(
    pages_df[
        ["document_title", "source_type", "page_number", "contains_legacy_engage_reference"]
    ].head(12)
)

## 4. Student organization directory

The notebook now includes a small verified starter directory so organization-discovery questions work immediately. It includes The Muslim Students' Association, Muslim Voice, the Muslim Law Students Association, and several other representative GW organizations.

Upload a complete official directory later to expand coverage. A loaded directory or future CampusGroups API data is merged with the starter records, and uploaded/current data takes priority.

In [ ]:
ORG_REQUIRED_COLUMNS = {"org_name", "category", "description"}

STARTER_ORGANIZATIONS = pd.DataFrame([
    {
        "org_id": "GW-MSA",
        "org_name": "The Muslim Students' Association",
        "category": "Religious, Secular, and Spiritual",
        "description": (
            "Serves Muslim students and the broader GW community through religious, "
            "social, cultural, educational, advocacy, and interfaith activities."
        ),
        "keywords": (
            "muslim islam islamic faith religion spiritual prayer jummah jumuah ramadan "
            "eid community interfaith msa"
        ),
        "source_url": "https://gwu.campusgroups.com/MSA/",
        "status": "active",
        "last_updated": "2026-07-20",
    },
    {
        "org_id": "GW-MUSLIM-VOICE",
        "org_name": "Muslim Voice",
        "category": "Religious, Secular, and Spiritual; Advocacy and Awareness",
        "description": (
            "Promotes understanding of Islam, interfaith dialogue, civic engagement, "
            "and the constructive representation of Islamic values."
        ),
        "keywords": (
            "muslim islam islamic interfaith dialogue civic engagement advocacy faith religion"
        ),
        "source_url": "https://gwu.campusgroups.com/MuslimVoice/",
        "status": "active",
        "last_updated": "2026-07-20",
    },
    {
        "org_id": "GW-LAW-MLSA",
        "org_name": "Muslim Law Students Association",
        "category": "Professional; Religious, Secular, and Spiritual",
        "description": (
            "Provides a religious, social, and professional community for Muslim and "
            "non-Muslim GW Law students interested in Islam and the Muslim legal community."
        ),
        "keywords": "muslim islam islamic law legal law school professional faith religion mlsa",
        "source_url": "https://gwu.campusgroups.com/MuslimLaw/",
        "status": "active",
        "last_updated": "2026-07-20",
    },
    {
        "org_id": "GW-ASA",
        "org_name": "African Students Association",
        "category": "Cultural",
        "description": (
            "Creates a welcoming space for the celebration of African cultures through "
            "academic, social, cultural, and service activities."
        ),
        "keywords": "africa african culture cultural diaspora social service community",
        "source_url": "https://gwu.campusgroups.com/AfricanSA/",
        "status": "active",
        "last_updated": "2026-07-20",
    },
    {
        "org_id": "GW-ISA",
        "org_name": "GW International Students Association",
        "category": "Cultural; Advocacy and Awareness",
        "description": (
            "Supports international students' adjustment, engagement, friendship, "
            "professional development, and connection to university resources."
        ),
        "keywords": "international global culture cultural friendship community students abroad",
        "source_url": "https://gwu.campusgroups.com/GWISA/",
        "status": "active",
        "last_updated": "2026-07-20",
    },
    {
        "org_id": "GW-ACM",
        "org_name": "GW Association for Computing Machinery Student Chapter",
        "category": "Academic; Professional",
        "description": (
            "Builds the computer science community through professor talks, game nights, "
            "technical learning, networking, and department-wide events."
        ),
        "keywords": "computer science computing coding programming software technology tech professional acm",
        "source_url": "https://gwu.campusgroups.com/GWACM/",
        "status": "active",
        "last_updated": "2026-07-20",
    },
    {
        "org_id": "GW-INDIAN-SA",
        "org_name": "Indian Students' Association",
        "category": "Cultural",
        "description": (
            "Promotes and celebrates Indian culture through educational, social, "
            "community-service, and cultural programs."
        ),
        "keywords": "india indian south asia culture cultural community service",
        "source_url": "https://gwu.campusgroups.com/ISA/",
        "status": "active",
        "last_updated": "2026-07-20",
    },
    {
        "org_id": "GW-IRANIAN-SA",
        "org_name": "Iranian Student Association",
        "category": "Cultural",
        "description": (
            "Builds community around Persian language and culture through social, "
            "educational, and cultural programs."
        ),
        "keywords": "iran iranian persian culture cultural language norooz community",
        "source_url": "https://gwu.campusgroups.com/IRSA/",
        "status": "active",
        "last_updated": "2026-07-20",
    },
    {
        "org_id": "GW-TURKISH-SA",
        "org_name": "GW Turkish Student Association",
        "category": "Cultural",
        "description": (
            "Connects the Turkish community with the wider GW community and promotes "
            "Turkish culture, communication, and cooperation."
        ),
        "keywords": "turkey turkish culture cultural international community",
        "source_url": "https://gwu.campusgroups.com/TurkishSA/",
        "status": "active",
        "last_updated": "2026-07-20",
    },
    {
        "org_id": "GW-FGU",
        "org_name": "GW First Gen United",
        "category": "Advocacy and Awareness",
        "description": (
            "Supports first-generation college students through community, advocacy, "
            "academic support, and professional development."
        ),
        "keywords": "first generation first-gen community advocacy academics professional support",
        "source_url": "https://gwu.campusgroups.com/FGU/",
        "status": "active",
        "last_updated": "2026-07-20",
    },
    {
        "org_id": "GW-PB",
        "org_name": "GW Program Board",
        "category": "Event Programming; University Spirit and Tradition",
        "description": (
            "Plans large campus programs and events intended to connect and entertain "
            "the broader GW community."
        ),
        "keywords": "events programming entertainment campus tradition community program board",
        "source_url": "https://gwu.campusgroups.com/GWPB/",
        "status": "active",
        "last_updated": "2026-07-20",
    },
    {
        "org_id": "GW-SGA",
        "org_name": "GW Student Government Association",
        "category": "Student Governance",
        "description": (
            "Represents the student body through advocacy, allocations, advertising, "
            "and assistance."
        ),
        "keywords": "student government governance advocacy finance allocations leadership sga",
        "source_url": "https://gwu.campusgroups.com/SGA/",
        "status": "active",
        "last_updated": "2026-07-20",
    },
])


def load_org_directory(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    elif path.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
    else:
        raise ValueError("Unsupported organization directory format.")

    df.columns = [str(column).strip().lower() for column in df.columns]
    missing = ORG_REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"Missing organization columns: {sorted(missing)}")

    for optional in ["org_id", "keywords", "source_url", "status", "last_updated"]:
        if optional not in df.columns:
            df[optional] = ""

    for column in df.columns:
        df[column] = df[column].fillna("").map(clean_text)

    df = df[
        ~df["org_name"].str.lower().str.startswith("replace with")
    ].copy()

    df = df[
        ~df["status"].str.lower().isin({"inactive", "archived", "deleted", "closed"})
    ].copy()

    return df.reset_index(drop=True)


directory_candidates = [
    path for path in source_files
    if path.suffix.lower() in {".csv", ".xlsx", ".xls"}
    and "organization" in path.name.lower()
]

uploaded_orgs_df = pd.DataFrame()

for candidate in directory_candidates:
    try:
        loaded = load_org_directory(candidate)
        if not loaded.empty:
            uploaded_orgs_df = loaded
            print("Loaded organization directory:", candidate.name)
            break
    except Exception as exc:
        print("Skipped", candidate.name, "-", exc)

# Merge the uploaded/current directory with the built-in starter directory.
# Uploaded data takes priority when organization names overlap.
if uploaded_orgs_df.empty:
    orgs_df = STARTER_ORGANIZATIONS.copy()
    directory_mode = "starter directory"
else:
    uploaded_names = set(uploaded_orgs_df["org_name"].str.lower())
    extra_starter_rows = STARTER_ORGANIZATIONS[
        ~STARTER_ORGANIZATIONS["org_name"].str.lower().isin(uploaded_names)
    ]
    orgs_df = pd.concat(
        [uploaded_orgs_df, extra_starter_rows],
        ignore_index=True,
    )
    directory_mode = "uploaded directory plus starter records"

print(f"Organization directory mode: {directory_mode}")
print(f"Organizations available for matching: {len(orgs_df):,}")
display(orgs_df[["org_name", "category", "description", "source_url"]].head(12))

# Preserve the bundled/uploaded directory as the fallback baseline before live API data is merged.
local_orgs_df = orgs_df.copy()


## 5. Structured CampusGroups, resource, funding, deadline, and budget data

In [ ]:
def find_data_file(filename: str) -> Path:
    matches = list(KNOWLEDGE_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Missing {filename}. Upload RevConnectAI_Knowledge_Base_v4.zip."
        )
    return matches[0]

campusgroups_df = pd.read_csv(find_data_file("campusgroups_howto.csv")).fillna("")
resources_df = pd.read_csv(find_data_file("campus_resources.csv")).fillna("")
funding_df = pd.read_csv(find_data_file("funding_programs.csv")).fillna("")
deadlines_df = pd.read_csv(find_data_file("deadline_registry.csv")).fillna("")
budget_assumptions_df = pd.read_csv(find_data_file("event_budget_assumptions.csv")).fillna("")
pathway_df = pd.read_csv(find_data_file("pathway_rules.csv")).fillna("")

PATHWAY_RULES = dict(zip(pathway_df["pathway"], pathway_df["funding_rule"]))
EVENT_BUDGET_ASSUMPTIONS = {
    "No food": {"low": 0.0, "typical": 0.0, "high": 0.0},
    "Snacks/refreshments": {"low": 6.0, "typical": 10.0, "high": 15.0},
    "Meal": {"low": 15.0, "typical": 22.0, "high": 32.0},
}

display(campusgroups_df[["topic", "audience", "source_url"]])
display(funding_df[["program", "best_for", "timing"]])

## 5. Live CampusGroups read-only connection

The prototype connects only to the official `rss_groups` and `rss_events` feeds. It requests published organizations and public upcoming events, excludes deleted/private records, stores normalized snapshots without credentials, and falls back to cached or bundled data if the connection is unavailable.

### Credential setup
In Google Colab, open **Secrets** (key icon) and add the credential as `CG_API_KEY` or `CG_API_SECRET`. Grant notebook access. The credential is sent only in the `X-CG-API-Secret` request header and is never printed, written to the notebook, or saved in the cache.

Because a credential was pasted into chat, rotate it in CampusGroups before production use and save the replacement in Colab Secrets.

In [ ]:
# This cell is generated from campusgroups_api_client.py by sync_notebook.py.
# Edit that file, then rerun the script -- do not edit this cell directly.

class CampusGroupsAPIError(RuntimeError):
    """Raised when a CampusGroups response cannot be retrieved or parsed."""


def clean_api_text(value: Any) -> str:
    if value is None:
        return ""
    text = re.sub(r"<[^>]+>", " ", str(value))
    return re.sub(r"\s+", " ", text).strip()


def local_name(tag: str) -> str:
    return str(tag).split("}")[-1]


def as_bool(value: Any) -> bool:
    return clean_api_text(value).casefold() in {"1", "true", "yes", "y", "on"}


def first_value(record: Mapping[str, Any], *names: str) -> str:
    lowered = {str(key).casefold(): value for key, value in record.items()}
    for name in names:
        value = lowered.get(name.casefold())
        if value not in (None, ""):
            return clean_api_text(value)
    return ""


def flatten_xml_record(node: ET.Element) -> Dict[str, str]:
    record: Dict[str, str] = {}
    for child in list(node):
        key = local_name(child.tag)
        if list(child):
            pieces = [clean_api_text(text) for text in child.itertext() if clean_api_text(text)]
            value = " | ".join(dict.fromkeys(pieces))
        else:
            value = clean_api_text(child.text)
        if key in record and value:
            record[key] = f"{record[key]} | {value}"
        else:
            record[key] = value
    return record


def parse_payload(content: bytes, expected_id_field: str) -> List[Dict[str, Any]]:
    stripped = content.lstrip()
    if not stripped:
        return []

    if stripped[:1] in {b"[", b"{"}:
        payload = json.loads(stripped.decode("utf-8"))
        if isinstance(payload, list):
            return [dict(item) for item in payload if isinstance(item, Mapping)]
        if isinstance(payload, Mapping):
            for key in ("items", "results", "data", "records"):
                value = payload.get(key)
                if isinstance(value, list):
                    return [dict(item) for item in value if isinstance(item, Mapping)]
            return [dict(payload)]

    try:
        root = ET.fromstring(content)
    except ET.ParseError as exc:
        preview = clean_api_text(content[:300].decode("utf-8", errors="replace"))
        raise CampusGroupsAPIError(f"CampusGroups returned non-XML data: {preview}") from exc

    expected = expected_id_field.casefold()
    candidates: List[ET.Element] = []
    for node in root.iter():
        child_names = {local_name(child.tag).casefold() for child in list(node)}
        if expected in child_names:
            candidates.append(node)

    if not candidates:
        for tag in ("item", "record", "row", "group", "event"):
            candidates.extend(root.findall(f".//{tag}"))

    records = [flatten_xml_record(node) for node in candidates]
    return [record for record in records if record]


@dataclass
class CampusGroupsConfig:
    # Left empty, base_url is derived from school_code. www.campusgroups.com is
    # the vendor's global tenant and does not serve GW records.
    base_url: str = ""
    groups_endpoint: str = "/rss_groups"
    events_endpoint: str = "/rss_events"
    school_code: str = "gwu"
    credential_names: Sequence[str] = ("CG_API_SECRET", "CG_API_KEY")
    timeout_seconds: int = 45
    event_future_days: int = 180
    event_limit: int = 2000
    cache_dir: Path = Path("/content/revconnect_ai/cache")
    user_agent: str = "RevConnectAI/5.0 (read-only CampusGroups integration)"
    # Event privacyLevel values the assistant may surface. 0 is public; 1 is the
    # authenticated campus community, which is RevConnectAI's audience. Confirm
    # the production meaning of each level with GW before widening this.
    allowed_event_privacy_levels: Sequence[str] = ("0", "1", "Everyone")

    def __post_init__(self) -> None:
        if not self.base_url:
            self.base_url = f"https://{self.school_code}.campusgroups.com"


@dataclass
class CampusGroupsStatus:
    enabled: bool
    connected: bool = False
    source: str = "none"
    message: str = "Not initialized"
    groups_count: int = 0
    events_count: int = 0
    refreshed_at_utc: str = ""
    endpoint_host: str = ""
    used_credential: bool = False
    warnings: List[str] = field(default_factory=list)

    def as_dict(self) -> Dict[str, Any]:
        return {
            "enabled": self.enabled,
            "connected": self.connected,
            "used_credential": self.used_credential,
            "source": self.source,
            "message": self.message,
            "groups_count": self.groups_count,
            "events_count": self.events_count,
            "refreshed_at_utc": self.refreshed_at_utc,
            "endpoint_host": self.endpoint_host,
            "warnings": list(self.warnings),
        }


class CampusGroupsReadOnlyClient:
    def __init__(self, config: Optional[CampusGroupsConfig] = None, credential: Optional[str] = None):
        self.config = config or CampusGroupsConfig()
        self.credential = (credential or self._load_credential() or "").strip()
        self.session = self._build_session()

    def _load_credential(self) -> Optional[str]:
        for name in self.config.credential_names:
            value = os.getenv(name)
            if value:
                return value.strip()
        try:
            from google.colab import userdata  # type: ignore
            for name in self.config.credential_names:
                try:
                    value = userdata.get(name)
                except Exception:
                    value = None
                if value:
                    return str(value).strip()
        except Exception:
            pass
        return None

    def _build_session(self) -> requests.Session:
        retry = Retry(
            total=3,
            connect=3,
            read=3,
            backoff_factor=0.8,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=frozenset({"GET"}),
            raise_on_status=False,
        )
        session = requests.Session()
        session.mount("https://", HTTPAdapter(max_retries=retry))
        session.headers.update({"User-Agent": self.config.user_agent, "Accept": "application/xml,text/xml,application/json"})
        return session

    @property
    def credential_available(self) -> bool:
        return bool(self.credential)

    def _get_records(self, endpoint: str, params: Mapping[str, Any], expected_id_field: str) -> List[Dict[str, Any]]:
        url = self.config.base_url.rstrip("/") + "/" + endpoint.lstrip("/")
        # The public feeds reject an invalid secret, so only send the header when
        # a credential is configured.
        headers = {"X-CG-API-Secret": self.credential} if self.credential else {}
        response = self.session.get(
            url,
            params={key: value for key, value in params.items() if value not in (None, "")},
            headers=headers,
            timeout=self.config.timeout_seconds,
        )
        if response.status_code in {401, 403}:
            if self.credential:
                raise CampusGroupsAPIError(
                    f"CampusGroups rejected the request with HTTP {response.status_code} while sending a credential. "
                    "These feeds are public and work without one: clear CG_API_KEY/CG_API_SECRET, or supply a valid secret."
                )
            raise CampusGroupsAPIError(
                f"CampusGroups returned HTTP {response.status_code} for {url}. "
                f"Confirm that the school code '{self.config.school_code}' is correct."
            )
        if not response.ok:
            raise CampusGroupsAPIError(f"CampusGroups request failed with HTTP {response.status_code}.")
        content_type = response.headers.get("content-type", "").casefold()
        if "html" in content_type and b"<html" in response.content[:500].lower():
            raise CampusGroupsAPIError("CampusGroups returned an HTML page instead of an API payload.")
        return parse_payload(response.content, expected_id_field=expected_id_field)

    def fetch_groups_raw(self) -> List[Dict[str, Any]]:
        return self._get_records(
            self.config.groups_endpoint,
            params={"include_unpublished": 0, "include_deleted": 0},
            expected_id_field="groupId",
        )

    def fetch_events_raw(self) -> List[Dict[str, Any]]:
        params: Dict[str, Any] = {
            "deleted": 0,
            "time_range": "upcoming_only",
            "future_day_range": self.config.event_future_days,
            "limit": self.config.event_limit,
            "privacy_displayed_to": 0,
        }
        # The feed honours privacy_level server-side. Only constrain it there
        # when the config asks for public-only events; otherwise the wider set
        # is fetched and filtered by normalize_events().
        allowed = {str(level) for level in self.config.allowed_event_privacy_levels}
        if not allowed - {"0", "Everyone"}:
            params["privacy_level"] = 0
        return self._get_records(
            self.config.events_endpoint,
            params=params,
            expected_id_field="eventUid",
        )


def normalize_groups(records: Iterable[Mapping[str, Any]]) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for record in records:
        deleted = as_bool(first_value(record, "deleted"))
        hidden = as_bool(first_value(record, "hidden"))
        published_raw = first_value(record, "published")
        published = True if not published_raw else as_bool(published_raw) or published_raw.casefold() == "published"
        if deleted or hidden or not published:
            continue

        name = first_value(record, "groupName", "name")
        if not name:
            continue
        mission = first_value(record, "mission")
        what_we_do = first_value(record, "whatWeDo")
        goals = first_value(record, "goals")
        description = clean_api_text(" ".join(part for part in (mission, what_we_do, goals) if part))
        # groupType is GW's funding-pathway label ("Student Organization (Blue
        # Line)"), not a topical category, so track whether a real category came
        # from the feed before falling back to it.
        category = first_value(record, "category")
        group_type = first_value(record, "groupType")
        has_source_category = bool(category)
        category = category or group_type
        acronym = first_value(record, "groupAcronym")
        # `closed` marks closed *membership* (joining needs approval), not a
        # defunct organization. It must not exclude the record.
        membership_closed = as_bool(first_value(record, "closed"))
        status = first_value(record, "groupStatus") or "active"
        source_url = first_value(record, "groupLink", "primaryWebSite", "webSite")
        has_source_description = bool(description)
        if not description:
            # Most live records carry no mission text, so build something the
            # embedding model can still match on instead of a bare name.
            description = clean_api_text(
                f"{name} is a GW {group_type or 'student organization'}"
                + (f" in the {category} category." if category else ".")
            )
        rows.append({
            "org_id": first_value(record, "groupId", "externalGroupId") or acronym or name,
            "org_name": name,
            "category": category or group_type or "Uncategorized",
            "description": description,
            # The name and acronym carry most of the searchable signal when the
            # mission fields are empty.
            "keywords": clean_api_text(f"{name} {acronym} {category} {group_type} {mission} {what_we_do} {goals}"),
            "source_url": source_url,
            "status": status,
            "last_updated": first_value(record, "lastUpdatedOn"),
            "group_acronym": acronym,
            "group_type": group_type,
            "group_logo_url": first_value(record, "groupLogoUrl"),
            "membership_closed": membership_closed,
            "has_source_description": has_source_description,
            "has_source_category": has_source_category,
            "data_source": "CampusGroups RSS API",
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=[
            "org_id", "org_name", "category", "description", "keywords", "source_url",
            "status", "last_updated", "group_acronym", "group_type", "group_logo_url",
            "membership_closed", "has_source_description", "has_source_category", "data_source"
        ])
    return df.drop_duplicates(subset=["org_id", "org_name"], keep="first").reset_index(drop=True)


def normalize_events(
    records: Iterable[Mapping[str, Any]],
    allowed_privacy_levels: Sequence[str] = ("0", "1", "Everyone"),
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    blocked_statuses = {"cancelled", "canceled", "rejected", "denied", "deleted"}
    allowed_privacy = {str(level) for level in allowed_privacy_levels}
    for record in records:
        if as_bool(first_value(record, "eventDelete", "deleted")):
            continue
        approval = first_value(record, "approvalStatus", "approvalStatus2")
        if approval.casefold() in blocked_statuses:
            continue
        title = first_value(record, "title", "eventName")
        if not title:
            continue
        privacy_level = first_value(record, "privacyLevel")
        privacy_displayed_to = first_value(record, "privacyDisplayedTo", "publishCalendar")
        if privacy_level and privacy_level not in allowed_privacy:
            continue
        if privacy_displayed_to and privacy_displayed_to not in {"0", "Everyone"}:
            continue
        rows.append({
            "event_id": first_value(record, "eventUid", "eventId", "externalEventId") or title,
            "event_title": title,
            "host_org": first_value(record, "group"),
            "host_org_id": first_value(record, "groupId", "groupCgId"),
            "group_acronym": first_value(record, "groupAcronym"),
            "category": first_value(record, "eventType", "groupType"),
            "description": first_value(record, "description"),
            "event_date": first_value(record, "eventDate"),
            "event_end_date": first_value(record, "eventEndDate"),
            "event_time": first_value(record, "eventTime"),
            "event_end_time": first_value(record, "eventEndTime"),
            "timezone": first_value(record, "timeZone", "timeZoneId"),
            "location": first_value(record, "eventLocation", "eventBuilding", "eventBuildingAddress"),
            "source_url": first_value(record, "eventLink", "iCalLink"),
            "visibility": "Everyone" if privacy_level in {"", "0", "Everyone"} else "GW campus community",
            "approval_status": approval,
            "attendance_estimate": first_value(record, "attendanceEstimate"),
            "last_updated": first_value(record, "lastUpdatedOn", "dateUpdated"),
            "data_source": "CampusGroups RSS API",
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=[
            "event_id", "event_title", "host_org", "host_org_id", "group_acronym", "category",
            "description", "event_date", "event_end_date", "event_time", "event_end_time", "timezone",
            "location", "source_url", "visibility", "approval_status", "attendance_estimate",
            "last_updated", "data_source"
        ])
    return df.drop_duplicates(subset=["event_id", "event_title"], keep="first").reset_index(drop=True)


def merge_organization_sources(local_df: pd.DataFrame, api_df: pd.DataFrame) -> pd.DataFrame:
    """Combine the curated directory with the live feed.

    The live feed is authoritative for which organizations exist and for their
    CampusGroups links, but it carries a mission for only a handful of records.
    So a live row wins on identity while curated description and keyword text is
    kept whenever the live record has none of its own.
    """
    if api_df.empty:
        return local_df.copy().reset_index(drop=True)
    local = local_df.copy()
    api = api_df.copy()
    for frame in (local, api):
        for column in ["org_id", "org_name", "category", "description", "keywords", "source_url", "status", "last_updated"]:
            if column not in frame.columns:
                frame[column] = ""
    for flag in ("has_source_description", "has_source_category"):
        if flag not in api.columns:
            api[flag] = False

    curated = {
        str(row["org_name"]).casefold(): row
        for _, row in local.iterrows()
        if str(row.get("org_name", "")).strip()
    }

    enriched_rows = []
    for _, row in api.iterrows():
        row = row.copy()
        match = curated.get(str(row["org_name"]).casefold())
        if match is not None:
            if not bool(row.get("has_source_description")) and str(match.get("description", "")).strip():
                row["description"] = match["description"]
            if not bool(row.get("has_source_category")) and str(match.get("category", "")).strip():
                row["category"] = match["category"]
            merged_keywords = f"{row.get('keywords', '')} {match.get('keywords', '')}".split()
            row["keywords"] = " ".join(dict.fromkeys(merged_keywords))
        enriched_rows.append(row)

    api_names = set(api["org_name"].fillna("").str.casefold())
    local_only = local[~local["org_name"].fillna("").str.casefold().isin(api_names)]
    merged = pd.DataFrame(enriched_rows) if enriched_rows else api
    return pd.concat([merged, local_only], ignore_index=True, sort=False).reset_index(drop=True)


def _cache_paths(config: CampusGroupsConfig) -> Tuple[Path, Path]:
    return config.cache_dir / "campusgroups_groups_snapshot.csv", config.cache_dir / "campusgroups_events_snapshot.csv"


def save_snapshots(groups_df: pd.DataFrame, events_df: pd.DataFrame, config: CampusGroupsConfig) -> None:
    config.cache_dir.mkdir(parents=True, exist_ok=True)
    groups_path, events_path = _cache_paths(config)
    groups_df.to_csv(groups_path, index=False)
    events_df.to_csv(events_path, index=False)


def load_snapshots(config: CampusGroupsConfig) -> Tuple[pd.DataFrame, pd.DataFrame]:
    groups_path, events_path = _cache_paths(config)
    groups = pd.read_csv(groups_path).fillna("") if groups_path.exists() else normalize_groups([])
    events = pd.read_csv(events_path).fillna("") if events_path.exists() else normalize_events([])
    return groups, events


def fetch_with_fallback(
    enabled: bool = True,
    config: Optional[CampusGroupsConfig] = None,
    credential: Optional[str] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, CampusGroupsStatus]:
    cfg = config or CampusGroupsConfig()
    status = CampusGroupsStatus(enabled=enabled, endpoint_host=cfg.base_url)
    if not enabled:
        groups, events = load_snapshots(cfg)
        status.source = "cache" if (not groups.empty or not events.empty) else "disabled"
        status.message = "Live CampusGroups refresh is disabled."
        status.groups_count = len(groups)
        status.events_count = len(events)
        return groups, events, status

    client = CampusGroupsReadOnlyClient(cfg, credential=credential)

    try:
        groups = normalize_groups(client.fetch_groups_raw())
        events = normalize_events(
            client.fetch_events_raw(),
            allowed_privacy_levels=cfg.allowed_event_privacy_levels,
        )
        save_snapshots(groups, events, cfg)
        status.connected = True
        status.used_credential = client.credential_available
        status.source = "live CampusGroups RSS API"
        status.message = (
            f"Connected successfully to {cfg.base_url} using the read-only group and event feeds"
            + (" with a configured credential." if client.credential_available else " (no credential required).")
        )
        status.groups_count = len(groups)
        status.events_count = len(events)
        status.refreshed_at_utc = datetime.now(timezone.utc).isoformat()
        return groups, events, status
    except Exception as exc:
        groups, events = load_snapshots(cfg)
        status.source = "cache" if (not groups.empty or not events.empty) else "local fallback"
        status.message = f"Live CampusGroups refresh failed; using fallback data. {type(exc).__name__}: {exc}"
        status.groups_count = len(groups)
        status.events_count = len(events)
        if client.credential_available:
            status.warnings.append(
                "A credential is configured. These feeds are public; an invalid secret causes a 403. "
                "Try clearing CG_API_KEY/CG_API_SECRET."
            )
        return groups, events, status


CAMPUSGROUPS_CONFIG = CampusGroupsConfig(
    school_code="gwu",
    cache_dir=CACHE_DIR,
    event_future_days=180,
    event_limit=2000,
)

api_orgs_df, api_events_df, campusgroups_status = fetch_with_fallback(
    enabled=USE_CAMPUSGROUPS_API,
    config=CAMPUSGROUPS_CONFIG,
)

# Live API rows take priority over the bundled starter directory when names overlap.
orgs_df = merge_organization_sources(local_orgs_df, api_orgs_df)
directory_mode = (
    "live CampusGroups API plus bundled fallback records"
    if campusgroups_status.connected
    else f"{campusgroups_status.source} plus bundled fallback records"
)

print("CampusGroups API status:", campusgroups_status.message)
print("Live/cached organizations:", len(api_orgs_df))
print("Live/cached public events:", len(api_events_df))
print("Organizations available for matching:", len(orgs_df))


## 6. Add organization records and engagement guidance

In [ ]:
def org_rows_to_records(df: pd.DataFrame) -> List[Dict[str, Any]]:
    records = []

    for index, row in df.iterrows():
        text = (
            f"Organization name: {row.get('org_name', '')}. "
            f"Category: {row.get('category', '')}. "
            f"Description: {row.get('description', '')}. "
            f"Keywords: {row.get('keywords', '')}. "
            f"Status: {row.get('status', '')}."
        )
        records.append({
            "record_id": stable_id("org", row.get("org_id", index), row.get("org_name", "")),
            "document_title": row.get("org_name", "Student Organization"),
            "filename": "student_organization_directory",
            "source_type": "Organization Directory",
            "authority_rank": 70,
            "source_url": row.get("source_url", ""),
            "document_updated": row.get("last_updated", ""),
            "page_number": None,
            "text": clean_text(text),
            "contains_legacy_engage_reference": False,
        })

    return records


def event_rows_to_records(df: pd.DataFrame) -> List[Dict[str, Any]]:
    records = []
    for index, row in df.iterrows():
        text = (
            f"Event: {row.get('event_title', '')}. "
            f"Host organization: {row.get('host_org', '')}. "
            f"Category: {row.get('category', '')}. "
            f"Date: {row.get('event_date', '')} {row.get('event_time', '')}. "
            f"Location: {row.get('location', '')}. "
            f"Description: {row.get('description', '')}."
        )
        records.append({
            "record_id": stable_id("event", row.get("event_id", index), row.get("event_title", "")),
            "document_title": row.get("event_title", "CampusGroups Event"),
            "filename": "campusgroups_public_events",
            "source_type": "CampusGroups Public Event",
            "authority_rank": 72,
            "source_url": row.get("source_url", ""),
            "document_updated": row.get("last_updated", ""),
            "page_number": None,
            "text": clean_text(text),
            "contains_legacy_engage_reference": False,
        })
    return records


ENGAGEMENT_GUIDANCE = [
    {
        "record_id": "engagement-recruitment",
        "document_title": "Student Organization Recruitment Guidance",
        "filename": "curated_engagement_guidance",
        "source_type": "Engagement Strategy",
        "authority_rank": 50,
        "source_url": "",
        "document_updated": datetime.now(timezone.utc).date().isoformat(),
        "page_number": None,
        "text": (
            "Recruitment suggestions include clearly explaining the organization's purpose, "
            "keeping its profile current, offering beginner-friendly events, collaborating with "
            "related organizations, following up with attendees, and providing several low-pressure "
            "ways for a new student to participate. These are recommendations, not policy."
        ),
        "contains_legacy_engage_reference": False,
    },
    {
        "record_id": "engagement-advertising",
        "document_title": "Student Organization Event Advertising Guidance",
        "filename": "curated_engagement_guidance",
        "source_type": "Engagement Strategy",
        "authority_rank": 50,
        "source_url": "",
        "document_updated": datetime.now(timezone.utc).date().isoformat(),
        "page_number": None,
        "text": (
            "Event advertising suggestions include using a specific title, explaining who the event "
            "is for and why it matters, including complete date time and location details, publishing "
            "early, sending reminders, coordinating with co-sponsors, and following up with attendees. "
            "Official policy and current platform instructions control required approvals."
        ),
        "contains_legacy_engage_reference": False,
    },
    {
        "record_id": "engagement-community",
        "document_title": "Student Organization Community-Building Guidance",
        "filename": "curated_engagement_guidance",
        "source_type": "Engagement Strategy",
        "authority_rank": 50,
        "source_url": "",
        "document_updated": datetime.now(timezone.utc).date().isoformat(),
        "page_number": None,
        "text": (
            "Community-building suggestions include consistent communication, personal welcomes, "
            "recurring small-group activities, meaningful member roles, recognition, feedback, "
            "and officer transition planning. Attendance alone should not be treated as a measure "
            "of a student's value or belonging."
        ),
        "contains_legacy_engage_reference": False,
    },
]
def structured_rows_to_records(
    df: pd.DataFrame,
    title_field: str,
    source_type: str,
    text_fields: Sequence[str],
) -> List[Dict[str, Any]]:
    records = []
    for index, row in df.iterrows():
        title = clean_text(row.get(title_field, source_type))
        text = " ".join(
            f"{field.replace('_', ' ').title()}: {clean_text(row.get(field, ''))}."
            for field in text_fields
            if clean_text(row.get(field, ""))
        )
        records.append({
            "record_id": stable_id(source_type, title, index),
            "document_title": title,
            "filename": source_type.lower().replace(" ", "_"),
            "source_type": source_type,
            "authority_rank": 75 if source_type != "Engagement Strategy" else 50,
            "source_url": row.get("source_url", ""),
            "document_updated": row.get("last_updated", ""),
            "page_number": None,
            "text": clean_text(text),
            "contains_legacy_engage_reference": False,
        })
    return records

all_records = page_records + ENGAGEMENT_GUIDANCE
if not orgs_df.empty:
    all_records.extend(org_rows_to_records(orgs_df))
if not api_events_df.empty:
    all_records.extend(event_rows_to_records(api_events_df))
all_records.extend(structured_rows_to_records(campusgroups_df, "topic", "CampusGroups How-To", ["audience", "summary", "steps"]))
all_records.extend(structured_rows_to_records(resources_df, "resource", "Campus Resource", ["keywords", "description", "contact"]))
all_records.extend(structured_rows_to_records(funding_df, "program", "Funding Guidance", ["best_for", "timing", "decision_timeline"]))
all_records.extend(structured_rows_to_records(deadlines_df, "topic", "Deadline and Timeline", ["deadline", "timeline", "status", "source"]))
records_df = pd.DataFrame(all_records)
print("Knowledge records:", len(records_df))

## 7. Chunk, embed, and index

In [ ]:
def chunk_text(text: str, chunk_size: int = 220, overlap: int = 45) -> List[str]:
    words = clean_text(text).split()
    if not words:
        return []

    chunks = []
    start = 0
    step = max(1, chunk_size - overlap)

    while start < len(words):
        end = min(len(words), start + chunk_size)
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        start += step

    return chunks


def build_chunks(records: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, record in records.iterrows():
        for chunk_index, chunk in enumerate(chunk_text(record["text"])):
            row = record.to_dict()
            row["chunk_id"] = stable_id(record["record_id"], chunk_index)
            row["chunk_index"] = chunk_index
            row["text"] = chunk
            rows.append(row)

    return pd.DataFrame(rows)


chunks_df = build_chunks(records_df)
if chunks_df.empty:
    raise RuntimeError("No chunks created.")

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

chunk_embeddings = embedding_model.encode(
    chunks_df["text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")

vector_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
vector_index.add(chunk_embeddings)

chunks_df.to_csv(CACHE_DIR / "knowledge_chunks.csv", index=False)
faiss.write_index(vector_index, str(CACHE_DIR / "knowledge.faiss"))

print("Indexed chunks:", len(chunks_df))

## 8. Intent routing, retrieval, and escalation

In [ ]:
HIGH_RISK_PATTERNS = [
    r"\bcontract\b", r"\bvendor\b", r"\breimburse", r"\bpayment\b",
    r"\bpay\b", r"\bfunding\b", r"\btravel\b", r"\balcohol\b",
    r"\bsafety\b", r"\bdiscriminat", r"\bharass", r"\bhazing\b",
    r"\baccessib", r"\blegal\b", r"\bemergency\b", r"\bapprove\b",
    r"\bapproval\b", r"\bappeal\b", r"\bexception\b",
    r"\bsign\b.*\bagreement\b", r"\bcashapp\b", r"\bcash app\b", r"\bvenmo\b",
]
ORG_DISCOVERY_NOUNS = ["organization", "organizations", "org", "orgs", "club", "clubs", "group", "groups", "association", "society"]

def looks_like_org_discovery(question: str) -> bool:
    q=question.lower()
    return any(n in q for n in ORG_DISCOVERY_NOUNS) and any(p in q for p in ["is there", "are there", "does gw have", "do we have", "on campus", "looking for", "find me", "which", "what org", "interested in", "recommend", "get involved"])

def classify_intent(question: str) -> str:
    q=question.lower()
    if looks_like_org_discovery(question): return "Organization Recommendation"
    if any(t in q for t in ["upcoming events", "events coming up", "events this week", "events today", "events tomorrow", "find an event", "what events", "public events", "things to do on campus"]): return "Event Discovery"
    if any(t in q for t in ["how much", "budget estimate", "cost estimate", "what will it cost", "event budget"]): return "Event Budget Estimate"
    if any(t in q for t in ["what funding", "how should we fund", "where should we get funding", "funding option", "general allocation or", "co-sponsorship or", "cosponsorship or", "fundraising or", "uwpf or"]): return "Funding Pathway"
    if any(t in q for t in ["help write", "what should i say", "funding application", "budget justification", "application narrative", "draft my request"]): return "Funding Application Coach"
    if any(t in q for t in ["deadline", "due date", "how long", "timeline", "when should", "when is", "hear back", "processing time"]): return "Deadline and Timeline"
    if any(t in q for t in ["campusgroups", "create an event", "update roster", "add members", "message members", "appoint officers", "check in", "rsvp", "group page", "send email", "create a form", "run an election"]): return "CampusGroups How-To"
    if any(t in q for t in ["resource", "who can help", "where can i go", "mental health", "career", "volunteer", "community service", "accommodation", "disability", "international student", "cultural support"]): return "Campus Resource"
    if any(t in q for t in ["advertise", "promote", "recruit", "retain members", "build community", "attendance", "turnout"]): return "Engagement Strategy"
    if any(t in q for t in ["sga bylaws", "finance committee", "general allocation", "co-sponsorship", "cosponsorship", "reclamation", "lbo", "uwpf"]): return "SGA Finance and Governance"
    return "Policy and General Guidance"

def needs_escalation(question: str) -> bool:
    q=question.lower(); return any(re.search(p,q) for p in HIGH_RISK_PATTERNS)

def domain_source_bonus(question: str, source_type: str) -> float:
    q=question.lower()
    if any(t in q for t in ["sga", "bylaw", "finance committee", "allocation", "co-sponsorship", "cosponsorship", "reclamation", "lbo", "uwpf"]): return 0.15 if source_type in {"SGA Governing Document", "SGA Guidance", "Funding Guidance"} else 0.0
    if any(t in q for t in ["contract", "travel", "reimbursement", "event policy", "registration", "officer", "membership", "hazing"]): return 0.15 if source_type=="University Policy" else 0.0
    if "campusgroups" in q or any(t in q for t in ["create an event", "group page", "add members", "appoint officers", "check in", "rsvp", "election"]): return 0.18 if source_type=="CampusGroups How-To" else 0.0
    if any(t in q for t in ["upcoming event", "events this week", "events today", "find an event"]): return 0.18 if source_type=="CampusGroups Public Event" else 0.0
    if any(t in q for t in ["deadline", "timeline", "how long", "hear back"]): return 0.18 if source_type=="Deadline and Timeline" else 0.0
    if any(t in q for t in ["career", "volunteer", "mental health", "accommodation"]): return 0.18 if source_type=="Campus Resource" else 0.0
    return 0.0

def retrieve(question: str, top_k: int=6, candidate_k: int=30) -> List[Dict[str,Any]]:
    qemb=embedding_model.encode([question],convert_to_numpy=True,normalize_embeddings=True).astype("float32")
    scores,indices=vector_index.search(qemb,min(candidate_k,len(chunks_df)))
    candidates=[]
    for semantic_score,index_value in zip(scores[0],indices[0]):
        if index_value<0: continue
        row=chunks_df.iloc[int(index_value)]
        adjusted=float(semantic_score)+float(row["authority_rank"])/1000.0+domain_source_bonus(question,row["source_type"])
        candidates.append({"semantic_score":float(semantic_score),"adjusted_score":adjusted,"document_title":row["document_title"],"filename":row["filename"],"source_type":row["source_type"],"authority_rank":int(row["authority_rank"]),"source_url":row["source_url"],"document_updated":row["document_updated"],"page_number":row["page_number"],"contains_legacy_engage_reference":bool(row["contains_legacy_engage_reference"]),"text":row["text"]})
    candidates.sort(key=lambda x:x["adjusted_score"],reverse=True)
    results=[]; seen=set()
    for c in candidates:
        ident=(c["document_title"],c["page_number"],c["text"][:80])
        if ident in seen: continue
        seen.add(ident); results.append(c)
        if len(results)>=top_k: break
    return results

## 9. Organization recommendation index

In [ ]:
org_index = None
org_search_texts: List[str] = []

ORG_STOPWORDS = {
    "a", "an", "and", "are", "at", "be", "can", "do", "does", "for", "have",
    "i", "in", "is", "it", "me", "my", "of", "on", "or", "student", "students",
    "the", "there", "to", "we", "with", "you", "your", "campus", "gw"
}


def normalize_tokens(text: str) -> set:
    tokens = re.findall(r"[a-z0-9]+", text.lower())
    return {token for token in tokens if token not in ORG_STOPWORDS and len(token) > 1}


def build_org_index(df: pd.DataFrame):
    global org_search_texts

    if df.empty:
        org_search_texts = []
        return None

    org_search_texts = (
        df["org_name"].fillna("") + " "
        + df["category"].fillna("") + " "
        + df["description"].fillna("") + " "
        + df["keywords"].fillna("")
    ).tolist()

    embeddings = embedding_model.encode(
        org_search_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index


if not orgs_df.empty:
    org_index = build_org_index(orgs_df)


def recommend_organizations(interests: str, top_k: int = 5) -> List[Dict[str, Any]]:
    if org_index is None or orgs_df.empty:
        return []

    query_embedding = embedding_model.encode(
        [interests],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    # Retrieve more candidates than will be displayed, then rerank them.
    candidate_count = min(max(top_k * 3, 10), len(orgs_df))
    semantic_scores, indices = org_index.search(query_embedding, candidate_count)

    query_tokens = normalize_tokens(interests)
    reranked = []

    for semantic_score, index_value in zip(semantic_scores[0], indices[0]):
        row = orgs_df.iloc[int(index_value)]
        org_text = org_search_texts[int(index_value)]
        org_tokens = normalize_tokens(org_text)

        overlap = len(query_tokens & org_tokens)
        lexical_bonus = min(overlap * 0.08, 0.32)

        exact_bonus = 0.0
        lowered_interest = interests.lower()
        lowered_org_text = org_text.lower()

        for important_term in [
            "muslim", "islam", "islamic", "law", "photography", "computer",
            "coding", "african", "indian", "iranian", "turkish",
            "international", "first generation", "government", "events"
        ]:
            if important_term in lowered_interest and important_term in lowered_org_text:
                exact_bonus += 0.18

        final_score = float(semantic_score) + lexical_bonus + exact_bonus

        reranked.append({
            "org_name": row["org_name"],
            "category": row["category"],
            "description": row["description"],
            "source_url": row.get("source_url", ""),
            "last_updated": row.get("last_updated", ""),
            "semantic_score": float(semantic_score),
            "match_score": final_score,
        })

    reranked.sort(key=lambda item: item["match_score"], reverse=True)
    return reranked[:top_k]

event_index = None
event_search_texts: List[str] = []

def build_event_index(df: pd.DataFrame):
    global event_search_texts
    if df.empty:
        event_search_texts = []
        return None
    event_search_texts = (
        df["event_title"].fillna("") + " "
        + df["host_org"].fillna("") + " "
        + df["category"].fillna("") + " "
        + df["description"].fillna("") + " "
        + df["location"].fillna("")
    ).tolist()
    embeddings = embedding_model.encode(
        event_search_texts, convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index

if not api_events_df.empty:
    event_index = build_event_index(api_events_df)

def recommend_events(interests: str, top_k: int = 5) -> List[Dict[str, Any]]:
    if event_index is None or api_events_df.empty:
        return []
    query = clean_text(interests) or "upcoming public events"
    query_embedding = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    candidate_count = min(max(top_k * 3, 10), len(api_events_df))
    semantic_scores, indices = event_index.search(query_embedding, candidate_count)
    query_tokens = normalize_tokens(query)
    reranked = []
    for semantic_score, index_value in zip(semantic_scores[0], indices[0]):
        row = api_events_df.iloc[int(index_value)]
        overlap = len(query_tokens & normalize_tokens(event_search_texts[int(index_value)]))
        reranked.append({
            "event_title": row.get("event_title", ""),
            "host_org": row.get("host_org", ""),
            "category": row.get("category", ""),
            "description": row.get("description", ""),
            "event_date": row.get("event_date", ""),
            "event_time": row.get("event_time", ""),
            "location": row.get("location", ""),
            "source_url": row.get("source_url", ""),
            "match_score": float(semantic_score) + min(overlap * 0.08, 0.32),
        })
    reranked.sort(key=lambda item: item["match_score"], reverse=True)
    return reranked[:top_k]

def format_api_status() -> str:
    status = campusgroups_status
    lines = [
        f"Enabled: {status.enabled}",
        f"Connected live: {status.connected}",
        f"Data source: {status.source}",
        f"Organizations: {status.groups_count}",
        f"Public upcoming events: {status.events_count}",
        f"Status: {status.message}",
    ]
    if status.refreshed_at_utc:
        lines.append(f"Refreshed UTC: {status.refreshed_at_utc}")
    if status.warnings:
        lines.append("Warnings: " + " | ".join(status.warnings))
    return "\n".join(lines)

def refresh_campusgroups_live() -> str:
    global api_orgs_df, api_events_df, campusgroups_status, orgs_df, org_index, event_index
    api_orgs_df, api_events_df, campusgroups_status = fetch_with_fallback(
        enabled=USE_CAMPUSGROUPS_API, config=CAMPUSGROUPS_CONFIG
    )
    orgs_df = merge_organization_sources(local_orgs_df, api_orgs_df)
    org_index = build_org_index(orgs_df)
    event_index = build_event_index(api_events_df)
    return format_api_status()

def live_org_search_table(query: str) -> pd.DataFrame:
    rows = recommend_organizations(query or "student organizations", top_k=10)
    if not rows:
        return pd.DataFrame(columns=["org_name", "category", "description", "source_url"])
    return pd.DataFrame(rows)[["org_name", "category", "description", "source_url"]]

def live_event_search_table(query: str) -> pd.DataFrame:
    rows = recommend_events(query or "upcoming public events", top_k=10)
    if not rows:
        return pd.DataFrame(columns=["event_title", "host_org", "event_date", "event_time", "location", "source_url"])
    return pd.DataFrame(rows)[["event_title", "host_org", "event_date", "event_time", "location", "source_url"]]


## 10. Simplified answer generation

The assistant now:

- answers the question in the first sentence;
- limits most responses to approximately 90 words;
- uses direct policy templates for common high-confidence questions;
- shows no more than two sources;
- avoids displaying the internal question category;
- avoids reproducing full handbook passages;
- gives organization matches directly for natural questions such as “Is there a Muslim student group?”

In [ ]:
_generator=None

def get_generator():
    global _generator
    if _generator is None:
        device=0 if torch.cuda.is_available() else -1
        _generator=pipeline("text2text-generation",model=LOCAL_GENERATION_MODEL,device=device)
    return _generator

def citation_label(item):
    title=item.get("document_title","Source"); page=item.get("page_number")
    return f"{title}, p. {int(float(page))}" if page is not None and str(page) not in {"", "nan", "None"} else title

def format_sources(results,source_limit=2):
    lines=[]; seen=set()
    for item in results:
        key=(item.get("document_title",""),item.get("page_number"),item.get("source_url",""))
        if key in seen: continue
        seen.add(key); label=citation_label(item); url=item.get("source_url","")
        lines.append(f"- {label}"+(f": {url}" if url else ""))
        if len(lines)>=source_limit: break
    return "\n".join(lines)

def build_context(results):
    return "\n\n".join(f"[Source {i}: {citation_label(x)} | {x['source_type']}]\n{x['text']}" for i,x in enumerate(results[:4],1))

def direct_policy_answer(question):
    q=question.lower()
    if ("contract" in q and "sign" in q) or re.search(r"\bcan\s+(i|we|a student|students)\b.*\bsign\b.*\bcontract\b",q):
        return "No. Students cannot sign contracts on behalf of a student organization or GW. Complete the Contract Information Sheet and send it with any vendor contract to your staff advisor or Org Help. Start at least 6–8 weeks before the event."
    if "venmo" in q or "cash app" in q or "cashapp" in q:
        return "No. Student organizations may not pay individuals through Venmo, Cash App, or similar mobile-payment apps, and those payments will not be reimbursed. Contact Org Help Finance before paying."
    if ("co-sponsorship" in q or "cosponsorship" in q) and any(t in q for t in ["how far","when","timeline","days","advance"]):
        return "Submit a co-sponsorship request at least 14 business days before the organization will incur the expense. Submit earlier when contracts or complex purchases are involved."
    if ("general allocation" in q or re.search(r"\bga\b",q)) and ("co-sponsorship" in q or "cosponsorship" in q) and any(t in q for t in ["difference","different","versus","vs"]):
        return "Use a General Allocation for planned future-semester programs, operations, or durable goods. Use a co-sponsorship for an eligible additional need or an on-campus event open to all GW students during the semester."
    if "reclamation" in q and any(t in q for t in ["what","why","mean","purpose"]):
        return "Reclamation returns unused SGA funds to the co-sponsorship pool so they can be reallocated to other organizations. It is not a punishment."
    return None

def campusgroups_howto_answer(question):
    q=question.lower()
    mappings=[(["create","event"],"Create an event"),(["group","page"],"Manage group page"),(["add","member"],"Add members"),(["appoint","officer"],"Appoint officers"),(["add","officer"],"Appoint officers"),(["email"],"Email and promote"),(["message"],"Email and promote"),(["check","attendance"],"Track attendance"),(["check in"],"Track attendance"),(["form"],"Forms, surveys, and elections"),(["survey"],"Forms, surveys, and elections"),(["election"],"Forms, surveys, and elections"),(["find","group"],"Find groups and events"),(["find","event"],"Find groups and events")]
    for terms,topic in mappings:
        if all(t in q for t in terms):
            row=campusgroups_df[campusgroups_df["topic"]==topic].iloc[0]
            return f"{row['summary']}\n\nSteps: {row['steps']}\n\nOfficial guidance: {row['source_url']}"
    return None

def recommend_resources(question,top_k=3):
    qt=normalize_tokens(question); scored=[]
    for _,row in resources_df.iterrows():
        overlap=len(qt & normalize_tokens(f"{row['resource']} {row['keywords']} {row['description']}"))
        if overlap>0: scored.append((overlap,row))
    scored.sort(key=lambda x:x[0],reverse=True)
    if not scored: return "I could not identify a specific resource. Start with Org Help for student-organization matters or the GW student resources directory."
    lines=["These resources are the closest matches:"]
    for _,row in scored[:top_k]: lines.append(f"- {row['resource']}: {row['description']} Contact: {row['contact']} Source: {row['source_url']}")
    return "\n".join(lines)

def deadline_answer(question):
    qt=normalize_tokens(question); scored=[]; qlow=question.lower()
    for _,row in deadlines_df.iterrows():
        overlap=len(qt & normalize_tokens(f"{row['topic']} {row['deadline']} {row['timeline']}"))
        if row['topic'] in qlow: overlap+=5
        if overlap>0: scored.append((overlap,row))
    scored.sort(key=lambda x:x[0],reverse=True)
    if not scored: return "Tell me the process—co-sponsorship, contract, re-registration, General Allocation, UWPF, new-organization recognition, or payment—and I will check the timeline."
    row=scored[0][1]
    return f"{row['topic'].title()}: {row['deadline']}. {row['timeline']} Status: {row['status']}.\n\nSource: {row['source']} — {row['source_url']}"

def funding_path_advisor(description,pathway="Unknown",open_to_all=True,campus_wide=False,future_semester=False,needs_flexible_funds=False):
    rec=[]
    if campus_wide: rec.append("University-Wide Programs Fund: strongest fit for a large campus-wide event, celebration, or tradition. Confirm the current cycle.")
    if future_semester: rec.append("SGA General Allocation: strongest fit for a planned future-semester program, recurring operation, or durable good.")
    if open_to_all and not future_semester: rec.append("SGA Co-Sponsorship: consider this for an on-campus event open to all GW students. Submit at least 14 business days before incurring the expense.")
    if needs_flexible_funds or not rec: rec.append("Fundraising or Revenue Funds: useful for flexible costs, reserves, items outside an SGA award, or needs too close to a funding deadline.")
    rec.append("Another student organization: a mission-aligned group may be able to co-sponsor or share costs.")
    pathway_note=PATHWAY_RULES.get(pathway,PATHWAY_RULES.get("Unknown","Confirm pathway eligibility."))
    return "Recommended funding paths:\n- "+"\n- ".join(rec[:4])+f"\n\nPathway note: {pathway_note}\n\nThis is planning guidance, not a funding decision. Confirm eligibility and current deadlines with SGA Finance or Org Help."

def funding_application_draft(event_name,organization_name,purpose,audience,expected_attendance,event_date,location,total_cost,amount_requested,funding_source,accessibility_plan):
    if not event_name or not purpose: return "Provide at least an event name and a clear purpose."
    total=float(total_cost or 0); requested=min(float(amount_requested or 0),total) if total else float(amount_requested or 0)
    return f"""Draft funding narrative\n\n{organization_name or 'Our organization'} requests ${requested:,.2f} from {funding_source} to support {event_name}, planned for {event_date or 'the proposed date'} at {location or 'the proposed location'}.\n\nPurpose and student benefit:\n{purpose}\n\nAudience and reach:\nThe program is intended for {audience or 'GW students'} and is expected to serve approximately {int(expected_attendance or 0)} participants. Explain how it advances the organization's mission and provides an educational, cultural, professional, service, or community-building benefit.\n\nBudget justification:\nThe total estimated cost is ${total:,.2f}. Attach an itemized budget and vendor quotes when available. Explain why each expense is necessary and identify revenue, dues, fundraising, co-sponsors, or existing funds.\n\nAccessibility and inclusion:\n{accessibility_plan or 'The organization will review accessibility, dietary, communication, and participation needs.'}\n\nDo not imply that funding is guaranteed. Confirm eligibility, deadlines, and allowable costs before submitting."""

def estimate_event_budget(attendees,food_option,supplies_per_person,marketing_cost,venue_cost,speaker_fee,performer_fee,av_cost,travel_cost,security_or_other_cost,contingency_percent):
    attendees=max(int(attendees or 0),0); a=EVENT_BUDGET_ASSUMPTIONS.get(food_option,EVENT_BUDGET_ASSUMPTIONS['No food'])
    fixed=sum(float(x or 0) for x in [marketing_cost,venue_cost,speaker_fee,performer_fee,av_cost,travel_cost,security_or_other_cost]); supplies=attendees*float(supplies_per_person or 0); estimates={}
    for level in ['low','typical','high']:
        subtotal=fixed+supplies+attendees*a[level]; estimates[level]=subtotal*(1+float(contingency_percent or 0)/100)
    return f"Planning estimate for {attendees} attendees:\n- Low: ${estimates['low']:,.2f}\n- Typical: ${estimates['typical']:,.2f}\n- High: ${estimates['high']:,.2f}\n\nFixed/manual costs entered: ${fixed:,.2f}\nSupplies: ${supplies:,.2f}\nFood assumption: {food_option}\nContingency: {float(contingency_percent or 0):.0f}%\n\nThis is a planning estimate, not a vendor quote or approved budget. Obtain actual quotes and confirm allowable costs."

def engagement_strategy_answer(question):
    q=question.lower()
    if 'advertis' in q or 'promot' in q or 'attendance' in q or 'turnout' in q: return "Create a complete CampusGroups event page with a clear title, benefit-focused description, date, time, location, image, and one RSVP call to action. Send a targeted email, post reminders, ask related organizations to co-promote, and follow up with attendees. Track registrations, attendance, opens, and repeat participation."
    if 'recruit' in q or 'more members' in q: return "Keep the group page current, explain what new members gain, host beginner-friendly events, use involvement fairs and co-hosted programs, and follow up personally with attendees. Give new members a small role quickly and create recurring ways to participate."
    if 'community' in q or 'retain' in q: return "Build community through consistent communication, recurring low-pressure gatherings, small-group interaction, meaningful member roles, recognition, feedback, and officer succession planning. Use data as a learning tool—not as a measure of a student's value."
    return "Use the CampusGroups group page, events, emails, forms, member tools, and attendance data together. Set one engagement goal, choose a specific audience, test a small action, and review the result."

def extractive_concise_fallback(question,results,max_sentences=3,max_words=95):
    qt=normalize_tokens(question); candidates=[]
    for rr,result in enumerate(results[:4]):
        for sr,sentence in enumerate(re.split(r"(?<=[.!?])\s+",clean_text(result['text']))):
            if len(sentence.split())<5: continue
            overlap=len(qt & normalize_tokens(sentence)); score=overlap*2.0-rr*0.25-sr*0.02
            if overlap>0: candidates.append((score,sentence.strip()))
    candidates.sort(key=lambda x:x[0],reverse=True); selected=[]; words=0; seen=set()
    for _,s in candidates:
        sig=s.lower()[:80]
        if sig in seen or words+len(s.split())>max_words: continue
        selected.append(s); seen.add(sig); words+=len(s.split())
        if len(selected)>=max_sentences: break
    return ' '.join(selected) if selected else "I found related information, but I could not verify a concise answer. Review the cited source or contact the responsible office."

def generate_grounded_answer(question,results,intent):
    direct=direct_policy_answer(question)
    if direct: return direct
    if not results: return "I could not verify this in the current knowledge base. Check the official source or contact the appropriate office."
    prompt=f"""You are RevConnectAI, a GW student support assistant. Use only the evidence. Start with the direct answer. Use plain language and no more than 90 words. Do not reproduce long passages. Do not invent policy, deadlines, organizations, CampusGroups steps, prices, or decisions. Funding and approval decisions remain with authorized staff.\nQuestion: {question}\nEvidence:\n{build_context(results)}\nConcise answer:"""
    try:
        response=get_generator()(prompt,max_new_tokens=125,do_sample=False,truncation=True); answer=clean_text(response[0]['generated_text'])
        if answer and len(answer.split())<=125: return answer
    except Exception: pass
    return extractive_concise_fallback(question,results)

def format_org_recommendations(question,recommendations):
    if not recommendations: return "I could not find a matching organization in the current directory. Try a broader interest or use the official CampusGroups directory."
    opening="Yes. Here are the strongest matches:" if re.search(r"\b(is there|are there|does gw have|do we have)\b",question.lower()) else "These organizations may match your interests:"
    lines=[opening]
    for i,item in enumerate(recommendations[:3],1): lines.append(f"{i}. {item['org_name']} ({item['category']}) — {item['description']} {item['source_url'] or 'Official link not available'}")
    return '\n\n'.join(lines)


def format_event_recommendations(question, recommendations):
    if not recommendations:
        return (
            "I could not retrieve matching public CampusGroups events. "
            "Check the official CampusGroups event directory or refresh the live connection."
        )
    lines = ["Here are the strongest matching upcoming public events:"]
    for i, item in enumerate(recommendations[:5], 1):
        when = clean_text(f"{item.get('event_date', '')} {item.get('event_time', '')}") or "Date/time not listed"
        host = item.get("host_org", "") or "Host not listed"
        location = item.get("location", "") or "Location not listed"
        link = item.get("source_url", "") or "Official link not available"
        lines.append(f"{i}. {item.get('event_title', 'Event')} — {when}; {location}; hosted by {host}. {link}")
    return "\n\n".join(lines)


def answer_question(question,top_k=5):
    question=clean_text(question)
    if not question: return "Enter a question."
    intent=classify_intent(question)
    if intent=='Organization Recommendation': return format_org_recommendations(question,recommend_organizations(question,top_k=5))
    if intent=='Event Discovery': return format_event_recommendations(question,recommend_events(question,top_k=5))
    if intent=='CampusGroups How-To':
        direct=campusgroups_howto_answer(question)
        if direct: return direct
    if intent=='Campus Resource': return recommend_resources(question)
    if intent=='Deadline and Timeline': return deadline_answer(question)
    if intent=='Funding Pathway': return "Use the Funding Pathway Advisor tab. General Allocation fits future-semester plans; co-sponsorship fits eligible open campus events; UWPF fits large campus-wide traditions; fundraising/revenue fits flexible needs."
    if intent=='Funding Application Coach': return "Use the Funding Application Coach tab. Explain student benefit, audience, mission connection, itemized costs, other funding, accessibility, and why each expense is necessary. Never imply approval is guaranteed."
    if intent=='Event Budget Estimate': return "Use the Event Budget Estimator tab and enter attendance, food, supplies, vendor fees, venue, A/V, travel, marketing, and other costs."
    if intent=='Engagement Strategy': return engagement_strategy_answer(question)
    results=retrieve(question,top_k=top_k); answer=generate_grounded_answer(question,results,intent); sections=[answer]
    if any(t in question.lower() for t in ['engage','campusgroups','platform','roster','event page']) and any(x['contains_legacy_engage_reference'] for x in results): sections.append('Note: a retrieved handbook passage refers to Engage. Use current CampusGroups instructions for platform steps.')
    if needs_escalation(question) and direct_policy_answer(question) is None: sections.append('Confirm this with Org Help, SGA Finance, or the responsible GW office before acting.')
    sources=format_sources(results,source_limit=2)
    if sources: sections.append('Sources:\n'+sources)
    return '\n\n'.join(sections)

## 11. Test questions

In [ ]:
TEST_QUESTIONS = [
    "Is there a Muslim student group on campus?",
    "How do I create an event in CampusGroups?",
    "What public events are coming up on campus?",
    "How can our organization advertise an event and recruit more members?",
    "I want to volunteer in DC but I am not in a student organization. Where should I go?",
    "Can I sign a contract for my student organization?",
    "What funding should we pursue for a large campus-wide cultural festival next semester?",
    "What is the co-sponsorship deadline and when should we hear back?",
    "How long can a vendor check take?",
    "What should I include in a funding application?",
    "How much should we budget for an event for 100 students?",
]
for q in TEST_QUESTIONS:
    print("="*100); print("QUESTION:",q); print(); print(answer_question(q)); print()

## 12. Starter retrieval evaluation

In [ ]:
evaluation_cases=pd.DataFrame([
{"question":"Is there a Muslim student group on campus?","expected_intent":"Organization Recommendation","expected_text":"The Muslim Students' Association"},
{"question":"How do I create an event in CampusGroups?","expected_intent":"CampusGroups How-To","expected_text":"Manage Group > Events > Create Event"},
{"question":"What public events are coming up on campus?","expected_intent":"Event Discovery","expected_text":""},
{"question":"Can I sign a contract for my student organization?","expected_intent":"Policy and General Guidance","expected_text":"Students cannot sign contracts"},
{"question":"What is the deadline for a co-sponsorship?","expected_intent":"Deadline and Timeline","expected_text":"14 business days"},
{"question":"Where can I find volunteer opportunities?","expected_intent":"Campus Resource","expected_text":"Honey W. Nashman Center"},
{"question":"How can we recruit more members?","expected_intent":"Engagement Strategy","expected_text":"beginner-friendly"},
])
for c in ['question','expected_intent','expected_text']: evaluation_cases[c]=evaluation_cases[c].fillna('').astype(str).str.strip()
def run_v5_evaluation(cases):
    rows=[]
    for _,case in cases.iterrows():
        q=str(case['question']); response=str(answer_question(q)); predicted=classify_intent(q)
        rows.append({'question':q,'expected_intent':str(case['expected_intent']),'predicted_intent':predicted,'intent_correct':predicted==str(case['expected_intent']),'expected_text':str(case['expected_text']),'behavior_correct':str(case['expected_text']).casefold() in response.casefold(),'response_word_count':len(response.split()),'response':response})
    return pd.DataFrame(rows)
evaluation_results=run_v5_evaluation(evaluation_cases)
evaluation_metrics=pd.DataFrame({'metric':['intent_accuracy','expected_behavior_accuracy','average_response_words'],'score':[evaluation_results['intent_correct'].mean(),evaluation_results['behavior_correct'].mean(),evaluation_results['response_word_count'].mean()]})
display(evaluation_results); display(evaluation_metrics)
evaluation_results.to_csv(CACHE_DIR/'v5_evaluation_results.csv',index=False)
evaluation_metrics.to_csv(CACHE_DIR/'v5_evaluation_metrics.csv',index=False)

## 13. Weekly refresh manifest

A production scheduler can rerun document and API ingestion weekly, rebuild the index, validate test questions, and publish the new version only if checks pass.

In [ ]:
refresh_manifest = {
    "refreshed_at_utc": datetime.now(timezone.utc).isoformat(),
    "pdf_files": [p.name for p in pdf_files],
    "pdf_pages": int(len(pages_df)),
    "knowledge_records": int(len(records_df)),
    "chunks": int(len(chunks_df)),
    "organization_records": int(len(orgs_df)),
    "campusgroups_api_enabled": bool(USE_CAMPUSGROUPS_API),
    "campusgroups_api_connected": bool(campusgroups_status.connected),
    "campusgroups_data_source": campusgroups_status.source,
    "campusgroups_groups": int(len(api_orgs_df)),
    "campusgroups_public_events": int(len(api_events_df)),
    "campusgroups_refreshed_at_utc": campusgroups_status.refreshed_at_utc,
}

(CACHE_DIR / "refresh_manifest.json").write_text(
    json.dumps(refresh_manifest, indent=2),
    encoding="utf-8",
)

refresh_manifest

## 14. Multi-tool demonstration interface

In [ ]:
# This cell is generated from revconnect_ui.py by sync_notebook.py.
# Edit that file, then rerun the script -- do not edit this cell directly.

import html
import re
from pathlib import Path

import gradio as gr

# A GW-styled monogram, not the university's trademarked logo. GW's official
# marks are controlled by Communications & Marketing and need approval; swap
# this file for the approved asset once that is granted.
def _find_gw_mark():
    """The notebook runs from the prototype folder locally and from /content in
    Colab, so the asset is looked up in both rather than assumed relative."""
    for base in (Path.cwd(), Path.cwd().parent, Path("/content")):
        candidate = base / "assets" / "gw_monogram.svg"
        if candidate.exists():
            return candidate
    return None


GW_MARK = _find_gw_mark()
GW_MARK_INLINE = (
    '<svg class="rc-mark" viewBox="0 0 64 64" role="img" aria-label="GW" focusable="false">'
    '<rect width="64" height="64" rx="12" fill="#033C5A"/>'
    '<rect y="55" width="64" height="9" fill="#D6BF91"/>'
    '<text x="32" y="42" font-family="Newsreader, Georgia, serif" font-size="30"'
    ' font-weight="600" text-anchor="middle" fill="#FFFFFF">GW</text>'
    '</svg>'
)
LAUNCH_KWARGS = {"favicon_path": str(GW_MARK)} if GW_MARK else {}

# --- Design tokens -----------------------------------------------------------
# GW's published brand palette (communications.gwu.edu/brand-guidelines/color-palette).
# GW Blue and GW Buff are the core pair, drawn from the Continental Army uniform;
# Potomac and Navy Yard are the sanctioned blue accents. Buff is light, so it
# works as a ground, a fill, and a rule on navy -- never as text on white.

GW_BLUE = "#033C5A"      # core
GW_BUFF = "#D6BF91"      # core
GW_BUFF_80 = "#DAC8A3"
GW_BUFF_50 = "#E8DDC6"
GW_BUFF_20 = "#F6F1E8"   # the page ground: warm, unmistakably GW
POTOMAC = "#0075C8"      # interactive blue
POTOMAC_50 = "#7FBAE3"
NAVY_YARD = "#00223E"    # ink, and the dark-mode ground
ROW_HOUSE = "#EF4343"    # secondary accent, used only for critical state
PATINA = "#ADCAB8"       # secondary accent, used only for healthy state

BUFF = gr.themes.Color(
    c50=GW_BUFF_20, c100="#F1E9DA", c200=GW_BUFF_50, c300=GW_BUFF_80,
    c400=GW_BUFF, c500="#C4A970", c600="#A98D55", c700="#856E41",
    c800="#5E4E2E", c900="#3D331E", c950="#221C10",
)
NAVY = gr.themes.Color(
    c50="#E9F3FB", c100="#CCE3F4", c200=POTOMAC_50, c300="#3391D3",
    c400=POTOMAC, c500="#005C9E", c600="#044A76", c700=GW_BLUE,
    c800="#022F47", c900=NAVY_YARD, c950="#001729",
)
# Warm-biased neutrals so the greys sit on the buff ground rather than fighting it.
STONE = gr.themes.Color(
    c50="#FAF8F4", c100="#F2EFE8", c200="#E4DED2", c300="#CFC7B7",
    c400="#A39C8E", c500="#7D7669", c600="#5E6670", c700="#454C55",
    c800="#2C333B", c900="#1A2129", c950="#0C1319",
)

THEME = gr.themes.Base(
    primary_hue=NAVY,
    secondary_hue=BUFF,
    neutral_hue=STONE,
    font=[gr.themes.GoogleFont("Public Sans"), "system-ui", "-apple-system", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("IBM Plex Mono"), "ui-monospace", "monospace"],
).set(
    body_background_fill=GW_BUFF_20,
    body_background_fill_dark="#001729",
    block_background_fill="#FFFFFF",
    block_background_fill_dark="#04283F",
    block_border_width="1px",
    block_border_color="#E4DED2",
    block_border_color_dark="#0B3A5C",
    block_radius="10px",
    block_label_text_weight="600",
    block_title_text_weight="600",
    body_text_color=NAVY_YARD,
    body_text_color_dark="#E8EFF5",
    body_text_color_subdued="#5E6670",
    body_text_color_subdued_dark="#9FB6C6",
    button_primary_background_fill=GW_BLUE,
    button_primary_background_fill_hover=POTOMAC,
    button_primary_background_fill_dark=GW_BUFF,
    button_primary_background_fill_hover_dark=GW_BUFF_80,
    button_primary_text_color="#FFFFFF",
    button_primary_text_color_dark=NAVY_YARD,
    button_large_radius="8px",
    button_small_radius="8px",
    input_background_fill="#FFFFFF",
    input_background_fill_dark="#012134",
    input_border_color="#E4DED2",
    input_border_color_dark="#0B3A5C",
    input_radius="8px",
)

# Gradio's theme only fetches the faces named in `font`/`font_mono`. Newsreader
# is used by the CSS below, so it has to be requested explicitly or it silently
# falls back to Georgia.
HEAD = (
    '<link rel="preconnect" href="https://fonts.googleapis.com">'
    '<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>'
    '<link rel="stylesheet" href="https://fonts.googleapis.com/css2'
    '?family=Newsreader:opsz,wght@6..72,500;6..72,600'
    '&amp;family=Public+Sans:wght@400;500;600;700'
    '&amp;family=IBM+Plex+Mono:wght@400;500&amp;display=swap">'
)

CSS = """
/* Only classes this file owns are styled here; the rest comes from the theme
   tokens above, so the design does not depend on Gradio's internal classes. */
:root {
  --rc-blue: #033C5A;        /* GW Blue */
  --rc-link: #0075C8;        /* Potomac */
  --rc-buff: #D6BF91;        /* GW Buff */
  --rc-ink: #00223E;         /* Navy Yard */
  --rc-muted: #5E6670;
  --rc-surface: #FFFFFF;
  --rc-border: #E4DED2;
  /* Semantic state, kept distinct from the core pair. Derived from GW's own
     secondary accents but darkened to hold contrast as text. */
  --rc-ok: #2F6B4F;   --rc-ok-bg: #EAF2ED;    /* Patina family */
  --rc-warn: #8A6D1F; --rc-warn-bg: #FFF8DC;  /* Parchment family */
  --rc-crit: #B02F2F; --rc-crit-bg: #FCEAEA;  /* Row House family */
  --rc-tag-bg: #F1E9DA; --rc-tag-fg: #6E5A33;   /* Buff tint for category tags */
}
/* Gradio stamps .dark on the wrapper for both the OS preference and its own
   toggle, so one signal covers both directions. */
.dark {
  --rc-blue: #7FBAE3;        /* Potomac 50% */
  --rc-link: #7FBAE3;
  --rc-buff: #D6BF91;
  --rc-ink: #E8EFF5;
  --rc-muted: #9FB6C6;
  --rc-surface: #04283F;
  --rc-border: #0B3A5C;
  --rc-ok: #ADCAB8;   --rc-ok-bg: #0C2E22;
  --rc-warn: #FFEFAE; --rc-warn-bg: #2E2814;
  --rc-crit: #EF4343; --rc-crit-bg: #331A1A;
  --rc-tag-bg: #0B3A5C; --rc-tag-fg: #DAC8A3;
}

/* fill_width is off, but Gradio still stretches the container on wide screens;
   the auto margins are what actually centre it. */
.gradio-container {
  max-width: 1180px !important;
  margin-left: auto !important;
  margin-right: auto !important;
}

/* Masthead ---------------------------------------------------------------- */
.rc-masthead {
  background: linear-gradient(180deg, #033C5A 0%, #00223E 100%); /* GW Blue -> Navy Yard */
  border-radius: 12px;
  padding: 22px 26px;
  display: flex;
  flex-wrap: wrap;
  align-items: center;
  justify-content: space-between;
  gap: 16px;
  border-bottom: 3px solid #D6BF91; /* GW Buff rule */
}
.rc-masthead__lockup { display: flex; align-items: flex-start; gap: 16px; min-width: 0; }
.rc-mark { width: 46px; height: 46px; flex: none; border-radius: 10px; margin-top: 2px; }
.rc-masthead__id { display: flex; flex-direction: column; gap: 5px; min-width: 0; }
.rc-eyebrow {
  font-size: 11px; font-weight: 600; letter-spacing: 0.09em;
  text-transform: uppercase; color: #D6BF91;
}
.rc-wordmark {
  font-family: Newsreader, Georgia, "Times New Roman", serif;
  font-size: 32px; font-weight: 600; line-height: 1.05; color: #FFFFFF;
  margin: 0; text-wrap: balance;
}
.rc-tagline { font-size: 14px; color: #CCE3F4; margin: 0; max-width: 62ch; } /* Potomac 20% */

/* Status chip ------------------------------------------------------------- */
.rc-chip {
  display: inline-flex; align-items: center; gap: 8px;
  padding: 7px 13px; border-radius: 999px;
  font-size: 12.5px; font-weight: 600; letter-spacing: 0.01em;
  border: 1px solid transparent; white-space: nowrap;
}
.rc-chip__dot { width: 8px; height: 8px; border-radius: 50%; flex: none; }
.rc-chip--ok   { background: var(--rc-ok-bg);   color: var(--rc-ok);   border-color: var(--rc-ok); }
.rc-chip--warn { background: var(--rc-warn-bg); color: var(--rc-warn); border-color: var(--rc-warn); }
.rc-chip--ok   .rc-chip__dot { background: var(--rc-ok); }
.rc-chip--warn .rc-chip__dot { background: var(--rc-warn); }
.rc-chip--proto {
  background: rgba(214,191,145,0.16); color: #E8DDC6; border-color: rgba(214,191,145,0.55);
}

/* Section headings -------------------------------------------------------- */
.rc-section h2 {
  font-family: Newsreader, Georgia, serif;
  font-size: 21px; font-weight: 600; color: var(--rc-ink);
  margin: 4px 0 3px; text-wrap: balance;
}
.rc-section p { font-size: 14px; color: var(--rc-muted); margin: 0; max-width: 68ch; }

/* Stat tiles -------------------------------------------------------------- */
.rc-tiles { display: flex; flex-wrap: wrap; gap: 10px; margin: 2px 0 6px; }
.rc-tile {
  flex: 1 1 130px; background: var(--rc-surface); border: 1px solid var(--rc-border);
  border-radius: 10px; padding: 12px 14px;
}
.rc-tile__label {
  font-size: 10.5px; font-weight: 600; letter-spacing: 0.08em;
  text-transform: uppercase; color: var(--rc-muted); margin-bottom: 5px;
}
.rc-tile__value {
  font-family: "IBM Plex Mono", ui-monospace, monospace;
  font-variant-numeric: tabular-nums;
  font-size: 22px; font-weight: 500; color: var(--rc-ink); line-height: 1.15;
}
.rc-tile__value--sm { font-size: 13px; }

/* Result tables ----------------------------------------------------------- */
/* Organization cards ------------------------------------------------------ */
.rc-cards {
  display: grid; gap: 12px;
  grid-template-columns: repeat(auto-fill, minmax(268px, 1fr));
  /* Most live records have no mission text, so let cards hug their content
     instead of stretching to the tallest in the row. */
  align-items: start;
}
.rc-card {
  display: flex; flex-direction: column; gap: 8px;
  background: var(--rc-surface); border: 1px solid var(--rc-border);
  border-radius: 10px; padding: 14px 16px;
}
.rc-card__name { font-size: 15px; font-weight: 600; margin: 0; line-height: 1.3; }
.rc-card__name, .rc-card__name a { color: var(--rc-ink); text-decoration: none; }
.rc-card__name a:hover { color: var(--rc-link); text-decoration: underline; }
.rc-card__desc {
  font-size: 13px; color: var(--rc-muted); line-height: 1.5; margin: 0;
  display: -webkit-box; -webkit-line-clamp: 3; -webkit-box-orient: vertical;
  overflow: hidden;
}
.rc-card__foot { margin-top: auto; padding-top: 2px; }
.rc-card__link { font-size: 12.5px; font-weight: 600; color: var(--rc-link); text-decoration: none; }
.rc-card__link:hover { text-decoration: underline; }
.rc-card__nolink { font-size: 12.5px; color: var(--rc-muted); }
.rc-tags { display: flex; flex-wrap: wrap; gap: 5px; }
.rc-tag {
  font-size: 10.5px; font-weight: 600; letter-spacing: 0.04em; text-transform: uppercase;
  background: var(--rc-tag-bg); color: var(--rc-tag-fg);
  border-radius: 4px; padding: 3px 7px; white-space: nowrap;
}

/* Event list -------------------------------------------------------------- */
.rc-events { display: flex; flex-direction: column; gap: 9px; }
.rc-event {
  display: flex; align-items: stretch; gap: 14px;
  background: var(--rc-surface); border: 1px solid var(--rc-border);
  border-radius: 10px; padding: 12px 16px;
}
.rc-event__date {
  flex: none; width: 52px; text-align: center;
  border-right: 1px solid var(--rc-border); padding-right: 12px;
  display: flex; flex-direction: column; justify-content: center;
}
.rc-event__mon {
  font-size: 10px; font-weight: 700; letter-spacing: 0.09em; color: var(--rc-blue);
}
.rc-event__day {
  font-family: "IBM Plex Mono", ui-monospace, monospace;
  font-variant-numeric: tabular-nums;
  font-size: 21px; font-weight: 500; color: var(--rc-ink); line-height: 1.1;
}
.rc-event__body { min-width: 0; display: flex; flex-direction: column; gap: 3px; }
.rc-event__title { font-size: 14.5px; font-weight: 600; margin: 0; line-height: 1.35; }
.rc-event__title, .rc-event__title a { color: var(--rc-ink); text-decoration: none; }
.rc-event__title a:hover { color: var(--rc-link); text-decoration: underline; }
.rc-event__meta { font-size: 12.5px; color: var(--rc-muted); }
.rc-event__where { font-size: 12px; }
.rc-empty {
  font-size: 13.5px; color: var(--rc-muted); font-style: italic;
  padding: 18px; border: 1px dashed var(--rc-border); border-radius: 10px; margin: 0;
}

/* Gradio's own footer badge is the one remaining non-GW mark. */
footer a[href*="gradio.app"] { display: none !important; }

/* Answer surface ---------------------------------------------------------- */
.rc-answer { min-height: 150px; }
.rc-answer p { line-height: 1.62; }
.rc-answer ol, .rc-answer ul { line-height: 1.62; }
.rc-answer a { color: var(--rc-link); text-underline-offset: 2px; }
.rc-answer blockquote {
  border-left: 3px solid var(--rc-warn); background: var(--rc-warn-bg);
  padding: 10px 14px; margin: 14px 0; border-radius: 0 8px 8px 0;
  color: var(--rc-ink); font-size: 14px;
}
.rc-answer .rc-placeholder { color: var(--rc-muted); font-style: italic; }
.rc-answer pre, .rc-answer table { overflow-x: auto; }

/* Footer ------------------------------------------------------------------ */
.rc-footer {
  border-top: 1px solid var(--rc-border); margin-top: 20px; padding-top: 14px;
  font-size: 12.5px; color: var(--rc-muted); line-height: 1.55;
}
.rc-footer strong { color: var(--rc-ink); }

*:focus-visible { outline: 2px solid var(--rc-buff); outline-offset: 2px; }
@media (prefers-reduced-motion: reduce) { * { transition: none !important; } }
@media (max-width: 700px) {
  .rc-wordmark { font-size: 26px; }
  .rc-masthead { padding: 18px; }
}
"""

# --- Presentation helpers ----------------------------------------------------

_URL_RE = re.compile(r"(?<![(<\"'])\bhttps?://[^\s<>\"')\]]+")

# Sentences the answer pipeline appends when a question needs staff sign-off or
# touches legacy platform wording. Surfaced as callouts so the safety behaviour
# is visible rather than buried in a wall of text.
_CALLOUT_PREFIXES = (
    "Confirm this with Org Help",
    "Note: a retrieved handbook passage refers to Engage",
)


def rc_markdown(text: str) -> str:
    """Render a plain-text answer as Markdown: live links, kept line breaks."""
    text = str(text or "").strip()
    if not text:
        return '<p class="rc-placeholder">Ask a question to see an answer here.</p>'

    blocks = []
    for block in re.split(r"\n\s*\n", text):
        block = block.strip()
        if not block:
            continue
        linked = _URL_RE.sub(lambda m: f"[{m.group(0)}]({m.group(0)})", block)
        if block.startswith(_CALLOUT_PREFIXES):
            blocks.append("> " + linked.replace("\n", "\n> "))
        else:
            # Two trailing spaces keeps single newlines as line breaks.
            blocks.append(linked.replace("\n", "  \n"))
    return "\n\n".join(blocks)


def rc_answer(question: str) -> str:
    return rc_markdown(answer_question(question))


def rc_deadline(question: str) -> str:
    return rc_markdown(deadline_answer(question))


def rc_funding_path(*args) -> str:
    return rc_markdown(funding_path_advisor(*args))


def rc_budget(*args) -> str:
    return rc_markdown(estimate_event_budget(*args))


def rc_application(*args) -> str:
    return rc_markdown(funding_application_draft(*args))


_MONTHS = ["JAN", "FEB", "MAR", "APR", "MAY", "JUN",
           "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"]


def _empty(message: str) -> str:
    return f'<p class="rc-empty">{html.escape(message)}</p>'


def _chips(value: str) -> str:
    """Categories arrive as 'Undergrad / Grad, Arts / Performance' -- split on
    the comma so each facet reads as its own tag."""
    parts = [p.strip() for p in str(value or "").split(",") if p.strip()]
    return "".join(f'<span class="rc-tag">{html.escape(p)}</span>' for p in parts[:3])


def _link(url: str, label: str) -> str:
    if not url:
        return '<span class="rc-card__nolink">No CampusGroups link</span>'
    return f'<a class="rc-card__link" href="{html.escape(url)}" target="_blank" rel="noopener">{label}</a>'


def rc_org_cards(query: str) -> str:
    """Organizations as cards. A dataframe of long names, category strings, and
    URLs reads like a spreadsheet export; cards let the name lead."""
    rows = recommend_organizations(query or "student organizations", top_k=12)
    if not rows:
        return _empty("No matching organizations. Try a broader interest.")
    cards = []
    for row in rows:
        raw_name = str(row.get("org_name", ""))
        name = html.escape(raw_name)
        url = str(row.get("source_url", "") or "")
        heading = f'<a href="{html.escape(url)}" target="_blank" rel="noopener">{name}</a>' if url else name

        # Records with no mission text get a synthesised description upstream
        # ("<name> is a GW <type> in the <category> category."), which only
        # repeats the heading and the tags. Drop it rather than pad the card.
        description = str(row.get("description", "") or "").strip()
        if description.startswith(f"{raw_name} is a GW"):
            description = ""
        body = f'<p class="rc-card__desc">{html.escape(description[:210])}</p>' if description else ""

        cards.append(
            '<article class="rc-card">'
            f'<h3 class="rc-card__name">{heading}</h3>'
            f'<div class="rc-tags">{_chips(row.get("category", ""))}</div>'
            f'{body}'
            f'<div class="rc-card__foot">{_link(url, "View on CampusGroups &rarr;")}</div>'
            '</article>'
        )
    return f'<div class="rc-cards">{"".join(cards)}</div>'


def _date_key(raw: str):
    """Sort key from M/D/YYYY. Undated rows sort last rather than first."""
    match = re.match(r"^(\d{1,2})/(\d{1,2})/(\d{4})", str(raw or "").strip())
    if not match:
        return (1, 0, 0, 0)
    month, day, year = (int(g) for g in match.groups())
    return (0, year, month, day)


def _date_block(raw: str) -> str:
    """M/D/YYYY -> a stacked month/day block; falls back to the raw string."""
    text = str(raw or "").strip()
    match = re.match(r"^(\d{1,2})/(\d{1,2})/(\d{4})", text)
    if not match:
        return f'<div class="rc-event__date"><div class="rc-event__day">--</div></div>'
    month, day, _ = (int(g) for g in match.groups())
    label = _MONTHS[month - 1] if 1 <= month <= 12 else ""
    return (
        '<div class="rc-event__date">'
        f'<div class="rc-event__mon">{label}</div>'
        f'<div class="rc-event__day">{day}</div>'
        '</div>'
    )


def rc_event_cards(query: str) -> str:
    rows = recommend_events(query or "upcoming public events", top_k=12)
    if not rows:
        return _empty("No matching upcoming events in the current feed.")
    # Relevance decides which events appear; a calendar should then read in
    # date order, not score order.
    rows = sorted(rows, key=lambda r: _date_key(r.get("event_date", "")))
    items = []
    for row in rows:
        title = html.escape(str(row.get("event_title", "Event")))
        url = str(row.get("source_url", "") or "")
        heading = f'<a href="{html.escape(url)}" target="_blank" rel="noopener">{title}</a>' if url else title
        host = html.escape(str(row.get("host_org", "") or "Host not listed"))
        location = html.escape(str(row.get("location", "") or "Location not listed"))
        time_text = html.escape(str(row.get("event_time", "") or ""))
        items.append(
            '<article class="rc-event">'
            + _date_block(row.get("event_date", ""))
            + '<div class="rc-event__body">'
            f'<h3 class="rc-event__title">{heading}</h3>'
            f'<div class="rc-event__meta">{host}</div>'
            f'<div class="rc-event__meta rc-event__where">{location}'
            + (f' &middot; {time_text}' if time_text else "")
            + '</div></div></article>'
        )
    return f'<div class="rc-events">{"".join(items)}</div>'


def rc_chip() -> str:
    """Connection state for the masthead, encoded in colour as well as words."""
    live = bool(campusgroups_status.connected)
    tone = "ok" if live else "warn"
    label = "Live CampusGroups data" if live else "Using bundled data"
    return (
        f'<span class="rc-chip rc-chip--{tone}">'
        f'<span class="rc-chip__dot"></span>{label}</span>'
    )


def rc_masthead() -> str:
    return (
        '<div class="rc-masthead">'
        '  <div class="rc-masthead__lockup">'
        f'  {GW_MARK_INLINE}'
        '  <div class="rc-masthead__id">'
        '    <div class="rc-eyebrow">The George Washington University</div>'
        '    <h1 class="rc-wordmark">RevConnectAI</h1>'
        '    <p class="rc-tagline">Student organization guidance, CampusGroups help, '
        'funding pathways, and event planning &mdash; grounded in GW policy documents '
        'and current CampusGroups listings.</p>'
        '  </div>'
        '  </div>'
        f'  <div style="display:flex;gap:8px;align-items:center;flex-wrap:wrap">{rc_chip()}'
        '    <span class="rc-chip rc-chip--proto">Prototype</span>'
        '  </div>'
        '</div>'
    )


def rc_status_panel() -> str:
    """Counts first, then the detail — the summary should read at a glance."""
    status = campusgroups_status
    live = bool(status.connected)
    refreshed = status.refreshed_at_utc.replace("T", " ")[:16] if status.refreshed_at_utc else "not yet"
    tiles = (
        '<div class="rc-tiles">'
        f'<div class="rc-tile"><div class="rc-tile__label">Organizations</div>'
        f'<div class="rc-tile__value">{status.groups_count:,}</div></div>'
        f'<div class="rc-tile"><div class="rc-tile__label">Upcoming events</div>'
        f'<div class="rc-tile__value">{status.events_count:,}</div></div>'
        f'<div class="rc-tile"><div class="rc-tile__label">Source</div>'
        f'<div class="rc-tile__value rc-tile__value--sm">{html.escape(status.source)}</div></div>'
        f'<div class="rc-tile"><div class="rc-tile__label">Refreshed (UTC)</div>'
        f'<div class="rc-tile__value rc-tile__value--sm">{html.escape(refreshed)}</div></div>'
        '</div>'
    )
    tone = "ok" if live else "warn"
    detail = (
        f'<div class="rc-chip rc-chip--{tone}" style="margin-bottom:10px">'
        f'<span class="rc-chip__dot"></span>'
        f'{"Connected" if live else "Not connected"} &middot; {html.escape(status.endpoint_host)}</div>'
        f'<p style="font-size:13.5px;color:var(--rc-muted);margin:0;line-height:1.55">'
        f'{html.escape(status.message)}</p>'
    )
    warnings = ""
    if status.warnings:
        items = "".join(f"<li>{html.escape(w)}</li>" for w in status.warnings)
        warnings = (
            f'<ul style="font-size:13px;color:var(--rc-warn);margin:10px 0 0;'
            f'padding-left:18px;line-height:1.5">{items}</ul>'
        )
    return tiles + detail + warnings


def rc_refresh():
    """Refresh live data, then restate connection status in both places it appears."""
    refresh_campusgroups_live()
    return rc_status_panel(), rc_masthead()


EXAMPLE_QUESTIONS = [
    "Is there a Muslim student group on campus?",
    "How do I create an event in CampusGroups?",
    "What public events are coming up on campus?",
    "Can I sign a contract for my student organization?",
    "What is the co-sponsorship deadline and when should we hear back?",
    "How can we recruit more members?",
]

DISCLAIMER = (
    '<div class="rc-footer"><strong>RevConnectAI provides guidance, not decisions.</strong> '
    'It does not approve funding, purchases, contracts, reimbursements, travel, or events. '
    'Current GW policy and authorized staff control. Verify anything you act on with Org Help, '
    'SGA Finance, or the responsible GW office. Organization and event listings come from public '
    'CampusGroups feeds; no student, membership, attendee, or payment data is requested.</div>'
)


def rc_section(title: str, blurb: str) -> str:
    return f'<div class="rc-section"><h2>{title}</h2><p>{blurb}</p></div>'


# --- Interface ---------------------------------------------------------------

with gr.Blocks(title="RevConnectAI", theme=THEME, css=CSS, head=HEAD, fill_width=False) as demo:
    masthead = gr.HTML(rc_masthead)

    with gr.Tabs():
        with gr.Tab("Ask"):
            gr.HTML(rc_section(
                "Ask a question",
                "Answers are grounded in the GW Student Organization Handbook, SGA bylaws and "
                "funding guidance, CampusGroups how-to steps, and live organization listings. "
                "High-risk topics are flagged for staff review.",
            ))
            with gr.Row():
                with gr.Column(scale=2, min_width=300):
                    question = gr.Textbox(
                        lines=4, label="Your question", show_label=True,
                        placeholder="e.g. What funding should we pursue for a cultural festival next semester?",
                    )
                    ask_button = gr.Button("Ask RevConnectAI", variant="primary")
                    gr.Examples(examples=EXAMPLE_QUESTIONS, inputs=question, label="Try one")
                with gr.Column(scale=3, min_width=340):
                    answer = gr.Markdown(
                        rc_markdown(""), label="Answer",
                        elem_classes=["rc-answer"], container=True, show_label=True,
                    )
            ask_button.click(rc_answer, question, answer)
            question.submit(rc_answer, question, answer)

        with gr.Tab("Organizations & events"):
            gr.HTML(rc_section(
                "Search current CampusGroups listings",
                "Registered organizations and upcoming events, read from GW's public "
                "CampusGroups feeds. Falls back to bundled data if the feed is unreachable.",
            ))
            status_panel = gr.HTML(rc_status_panel)
            with gr.Row():
                refresh_button = gr.Button(
                    "Refresh from CampusGroups", variant="secondary",
                    size="sm", scale=0, min_width=220,
                )

            # Tables get the full width; side by side they clip their columns.
            with gr.Row():
                org_query = gr.Textbox(
                    label="Find organizations", scale=4, min_width=240,
                    placeholder="photography, public service, coding, cultural community",
                )
                org_button = gr.Button("Search", variant="primary", scale=0, min_width=130)
            org_results = gr.HTML(lambda: rc_org_cards(""), label="Matching organizations")

            with gr.Row():
                event_query = gr.Textbox(
                    label="Find events", scale=4, min_width=240,
                    placeholder="volunteer, career, cultural, social, technology",
                )
                event_button = gr.Button("Search", variant="primary", scale=0, min_width=130)
            event_results = gr.HTML(lambda: rc_event_cards(""), label="Upcoming events")
            refresh_button.click(rc_refresh, outputs=[status_panel, masthead])
            org_button.click(rc_org_cards, org_query, org_results)
            org_query.submit(rc_org_cards, org_query, org_results)
            event_button.click(rc_event_cards, event_query, event_results)
            event_query.submit(rc_event_cards, event_query, event_results)

        with gr.Tab("Funding pathway"):
            gr.HTML(rc_section(
                "Which funding source fits",
                "Compares SGA General Allocation, co-sponsorship, the University-Wide Programs "
                "Fund, and self-generated funds against what you are planning. Planning guidance "
                "only — eligibility and deadlines are confirmed by SGA Finance.",
            ))
            with gr.Row():
                with gr.Column(min_width=300):
                    need = gr.Textbox(lines=4, label="What are you planning?")
                    pathway = gr.Dropdown(
                        ["Unknown", "Red", "Blue", "Orange", "Silver", "Yellow", "Green"],
                        value="Unknown", label="Organization pathway",
                    )
                    open_all = gr.Checkbox(True, label="Open to all GW students")
                    campuswide = gr.Checkbox(False, label="Large campus-wide event or tradition")
                    future = gr.Checkbox(False, label="Happens in a future semester")
                    flexible = gr.Checkbox(False, label="Needs flexible or self-generated funds")
                    funding_button = gr.Button("Recommend a pathway", variant="primary")
                with gr.Column(min_width=340):
                    funding_out = gr.Markdown(
                        rc_markdown(""), label="Recommendation",
                        elem_classes=["rc-answer"], container=True, show_label=True,
                    )
            funding_button.click(
                rc_funding_path,
                [need, pathway, open_all, campuswide, future, flexible],
                funding_out,
            )

        with gr.Tab("Budget estimate"):
            gr.HTML(rc_section(
                "Estimate an event budget",
                "Produces low, typical, and high planning figures from GW cost assumptions. "
                "A planning estimate, not a quote or an approved budget.",
            ))
            with gr.Row():
                with gr.Column(min_width=300):
                    with gr.Row():
                        attendees = gr.Number(50, label="Expected attendees")
                        food = gr.Dropdown(
                            list(EVENT_BUDGET_ASSUMPTIONS.keys()),
                            value="Snacks/refreshments", label="Food",
                        )
                    with gr.Row():
                        supplies = gr.Number(5, label="Supplies per person ($)")
                        marketing = gr.Number(100, label="Marketing ($)")
                    with gr.Accordion("Vendor, venue, and travel costs", open=False):
                        with gr.Row():
                            venue = gr.Number(0, label="Venue ($)")
                            av = gr.Number(0, label="A/V ($)")
                        with gr.Row():
                            speaker = gr.Number(0, label="Speaker fee ($)")
                            performer = gr.Number(0, label="Performer or DJ fee ($)")
                        with gr.Row():
                            travel = gr.Number(0, label="Travel ($)")
                            other = gr.Number(0, label="Security or other ($)")
                    contingency = gr.Number(10, label="Contingency (%)")
                    budget_button = gr.Button("Estimate budget", variant="primary")
                with gr.Column(min_width=340):
                    budget_out = gr.Markdown(
                        rc_markdown(""), label="Planning estimate",
                        elem_classes=["rc-answer"], container=True, show_label=True,
                    )
            budget_button.click(
                rc_budget,
                [attendees, food, supplies, marketing, venue, speaker,
                 performer, av, travel, other, contingency],
                budget_out,
            )

        with gr.Tab("Application draft"):
            gr.HTML(rc_section(
                "Draft a funding request",
                "Turns your event details into a narrative covering student benefit, audience, "
                "budget justification, and accessibility. Review and edit before submitting; "
                "a draft never implies approval.",
            ))
            with gr.Row():
                with gr.Column(min_width=300):
                    with gr.Row():
                        event_name = gr.Textbox(label="Event or project")
                        org_name = gr.Textbox(label="Organization")
                    purpose = gr.Textbox(lines=4, label="Purpose and student benefit")
                    with gr.Row():
                        audience = gr.Textbox(label="Audience")
                        attendance = gr.Number(50, label="Expected attendance")
                    with gr.Row():
                        event_date = gr.Textbox(label="Date")
                        location = gr.Textbox(label="Location")
                    with gr.Row():
                        total = gr.Number(0, label="Total cost ($)")
                        requested = gr.Number(0, label="Amount requested ($)")
                    source = gr.Dropdown(
                        ["SGA General Allocation", "SGA Co-Sponsorship",
                         "University-Wide Programs Fund", "Other"],
                        value="SGA Co-Sponsorship", label="Funding source",
                    )
                    accessibility = gr.Textbox(lines=3, label="Accessibility and inclusion plan")
                    draft_button = gr.Button("Draft the request", variant="primary")
                with gr.Column(min_width=340):
                    draft_out = gr.Markdown(
                        rc_markdown(""), label="Draft narrative",
                        elem_classes=["rc-answer"], container=True, show_label=True,
                    )
            draft_button.click(
                rc_application,
                [event_name, org_name, purpose, audience, attendance, event_date,
                 location, total, requested, source, accessibility],
                draft_out,
            )

        with gr.Tab("Deadlines"):
            gr.HTML(rc_section(
                "Check a deadline",
                "Co-sponsorship, contracts, re-registration, General Allocation, UWPF, "
                "new-organization recognition, and payment timelines.",
            ))
            with gr.Row():
                with gr.Column(min_width=300):
                    process = gr.Textbox(
                        lines=2, label="Which process?",
                        placeholder="e.g. co-sponsorship, vendor check, re-registration",
                    )
                    deadline_button = gr.Button("Check the timeline", variant="primary")
                with gr.Column(min_width=340):
                    deadline_out = gr.Markdown(
                        rc_markdown(""), label="Deadline and timeline",
                        elem_classes=["rc-answer"], container=True, show_label=True,
                    )
            deadline_button.click(rc_deadline, process, deadline_out)
            process.submit(rc_deadline, process, deadline_out)

    gr.HTML(DISCLAIMER)

demo.launch(share=True, debug=False, **LAUNCH_KWARGS)


## Production checklist
- Rotate the credential that was pasted into chat and store the replacement only in an approved secret manager.
- Confirm the credential is an API secret permitted to use CampusGroups RSS feeds.
- Keep the integration read-only unless GW approves a separate write-service design and authorization process.
- Maintain the public-event filters: `privacy_level=0`, `privacy_displayed_to=0`, `deleted=0`, and do not override location privacy.
- Do not enable user, membership, attendee, payment, transaction, or private-event feeds in the student-facing assistant.
- Replace remaining legacy Engage URLs in the starter directory with live CampusGroups links.
- Validate organization types/categories and public-event visibility against GW's production configuration.
- Schedule a weekly refresh, rebuild the vector index, run the evaluation suite, and publish only after checks pass.
- Add authentication, authorization, audit logging, rate limiting, monitoring, and an incident-response process before deployment.
- Keep policy citations separate from live operational data; CampusGroups data does not create or override GW policy.
- Obtain privacy, security, accessibility, records-retention, and institutional approval before production use.